# 02 · Alt Küme ve Temizlik
Amaç: 01'deki bulgulara göre hangi oyunları ve yorumları kullanacağımıza karar vermek, veriyi temizleyip `data/processed/` altına kaydetmek.
Kaynaklar: `games.json` (oyun bilgileri ve oylar), `reviews/*.csv` (yorumlar), 01'in ara çıktıları (iki parquet).

In [1]:
import json
from pathlib import Path

import pandas as pd

PROJE = Path.home() / "steam-oyun-asistani"
HAM = PROJE / "data" / "raw"
ISLENMIS = PROJE / "data" / "processed"
YORUM_KLASORU = HAM / "reviews"

In [2]:
with open(HAM / "games.json", encoding="utf-8") as f:
    oyunlar = json.load(f)

dosya_ozeti = pd.read_parquet(ISLENMIS / "yorum_dosya_ozeti.parquet")
ornek = pd.read_parquet(ISLENMIS / "yorum_ornek_yuzde1.parquet")

print("games.json oyun sayısı:", len(oyunlar))
print("yorum_dosya_ozeti:", dosya_ozeti.shape, "| index tipi:", dosya_ozeti.index.dtype)
print(dosya_ozeti.columns.tolist())
print("yorum_ornek_yuzde1:", ornek.shape)
print(ornek.columns.tolist())

games.json oyun sayısı: 65686
yorum_dosya_ozeti: (23107, 9) | index tipi: str
['satir', 'recommended', 'not_recommended', 'recommend_diger', 'bos_review', 'uzunluk_medyan', 'uzunluk_max', 'helpful_0', 'early_access_dolu']
yorum_ornek_yuzde1: (317072, 7)
['playtime', 'helpfulness', 'review', 'recommend', 'app_id', 'dil', 'dil_guven']


### Yüklenen veriler
- `oyunlar` (games.json): 65.686 oyun, sözlük, anahtar app_id (metin).
- `dosya_ozeti` (01'de CSV'lerden çıkarılan özet): 23.107 satır, her CSV dosyası için bir satır. Index app_id ve tipi **str**, yani games.json ile doğrudan eşleşiyor.
  Sütunlar: satir, recommended, not_recommended, recommend_diger, bos_review, uzunluk_medyan, uzunluk_max, helpful_0, early_access_dolu.
- `ornek` (CSV'lerden %1 örnek): 317.072 yorum. Dil sütunları (dil, dil_guven) 01'de eklendi. user / post_date / early_access_review bu örnekte yok.

games.json   → 65.686 oyun  (hepsinin bilgisi var)
  └ bunların 23.107'sinin yorum CSV'si var → dosya_ozeti'nde 23.107 satır

## Karar 1: Alt küme kuralı
Soru: Hangi oyunları kullanacağız? Kural: "CSV dosyasında en az X yorum olan oyunlar".
Ölçüm: Farklı X değerlerinde kalan oyun ve yorum sayısı (kaynak: dosya_ozeti["satir"], yani CSV'lerdeki yorum sayısı).

In [3]:
esikler = [0, 50, 100, 200, 500, 1000, 5000]
toplam_yorum = dosya_ozeti["satir"].sum()

satirlar = []
for esik in esikler:
    secilen = dosya_ozeti[dosya_ozeti["satir"] >= esik]
    satirlar.append({
        "esik_csv_yorum": esik,
        "oyun_sayisi": len(secilen),
        "yorum_sayisi": secilen["satir"].sum(),
        "yorum_payi_yuzde": round(secilen["satir"].sum() / toplam_yorum * 100, 1),
    })

esik_tablosu = pd.DataFrame(satirlar)
esik_tablosu

,esik_csv_yorum,oyun_sayisi,yorum_sayisi,yorum_payi_yuzde
0,0,23107,31832390,100.0
1,50,16503,31597946,99.3
2,100,11576,31252405,98.2
3,200,8005,30751163,96.6
4,500,4818,29740667,93.4
5,1000,3226,28610592,89.9
6,5000,1064,23837816,74.9


**Bulgu:** Yorumlar az sayıda popüler oyunda toplanmış. CSV'de ≥100 yorum şartı oyunların yarısını atıyor ama yorumların %98,2'si kalıyor; ≥5000 yorumlu 1.064 oyun tüm yorumların %75'ini taşıyor.
**Sonuç:** Oyun eşiği yorum sayısını (≈31 M) küçültmüyor. Yorum hacmi ayrı bir kararla (oyun başına üst sınır / kısa yorum eşiği) çözülecek.
**Karar 1:** Yorum CSV'si olan 23.107 oyunun hepsi alınır (eşik yok). Az yorumlu oyunlar için SQL'de `yorum_sayisi` sütunu tutulur; agent az yorumlu oyunlarda kullanıcıyı uyarır (30 soruda test edilecek).

In [4]:
csvli = set(dosya_ozeti.index)
yorumsuz = [app_id for app_id in oyunlar if app_id not in csvli]

yorumsuz_tablo = pd.DataFrame({
    "ad": [oyunlar[a]["name"] for a in yorumsuz],
    "oy": [oyunlar[a]["positive"] + oyunlar[a]["negative"] for a in yorumsuz],
}, index=yorumsuz)

print("CSV'si olmayan oyun:", len(yorumsuz_tablo))
print("games.json'da 0 oylu:", (yorumsuz_tablo["oy"] == 0).sum())
print("games.json'da 1-99 oylu:", yorumsuz_tablo["oy"].between(1, 99).sum())
print("games.json'da 100+ oylu:", (yorumsuz_tablo["oy"] >= 100).sum())
yorumsuz_tablo.sort_values("oy", ascending=False).head(20)

CSV'si olmayan oyun: 42579
games.json'da 0 oylu: 14984
games.json'da 1-99 oylu: 27275
games.json'da 100+ oylu: 320


,ad,oy
1426210,It Takes Two,180116
1716740,Starfield,160947
1274570,DEVOUR,79359
1290000,PowerWash Simulator,48209
1206560,WorldBox - God Simulator,37218
1062090,Timberborn,27568
1448440,Wo Long: Fallen Dynasty,26653
1948980,Hero's Adventure: Road to Passion,19739
1096530,Solasta: Crown of the Magister,19385
1466640,Road 96 🛣️,19291


### Ara soru: CSV'si olmayan oyunlar SQL'e girsin mi?
games.json'da olup yorum CSV'si olmayan 42.579 oyun var: 14.984'ü 0 oylu, 27.275'i 1–99 oylu, **320'si 100+ oylu**.
Popüler olup CSV'si hiç olmayan oyunlar var: It Takes Two (180.116 oy), Starfield (160.947), DEVOUR, PowerWash Simulator, CoD: MW III.
Bu, Sims 4'teki "CSV var ama eksik" durumundan farklı: burada yorum hiç yok.

**Karar 1 (güncel):**
- **SQL:** games.json'daki 65.686 oyunun hepsi. Her oyuna üç yardımcı sütun eklenecek:
  - `oy_sayisi` = games.json'daki positive + negative
  - `yorum_var` = yorum CSV'si var mı (True/False)
  - `iskelet` = açıklama, tür, dil ve kategori alanlarının hepsi boş mu (True/False)
- **Chroma (yorumlar):** yorum CSV'si olan 23.107 oyunun hepsi, eşik yok. (Yorum hacmi, ≈31,8 M, ayrı kararla küçültülecek.)
- **Agent kuralları (07'de):**
  - `yorum_var = False` ise "bu oyunun yorumları veri setinde yok" der, sayısal bilgileri yine verir.
  - Az yorumlu oyunlarda yorum özetini "az sayıda yoruma dayanıyor" diye uyarır.
  - Sıralama/listeleme sorularında kullanıcı aksini istemedikçe az oylu oyunları eler (örneğin `oy_sayisi` ≥ 100).
- **30 soruya aday testler:** "Starfield hakkında oyuncular ne diyor?" (doğru cevap: yorum yok), az yorumlu bir oyun (uyarı veriyor mu?), "en yüksek puanlı korku oyunları" (az oylu oyunları eliyor mu?).

Neden: Hedef asistanın bütün kabiliyetini göstermek. SQL'de ek oyun maliyetsiz; riskler (yorumsuz oyun, gürültülü oran) sütunlar ve agent kurallarıyla yönetiliyor.

## Karar 2: Oran kaynağı
Olumlu oran iki yerden hesaplanabilir: games.json (positive/negative) veya CSV'ler (recommend sütunu, dosya_ozeti'ndeki recommended/not_recommended).
Önce her oyunda hangi kaynağın oy içerdiğini sayıyoruz.

In [5]:
oy_karsilastirma = pd.DataFrame({
    "ad": [oyunlar[a]["name"] for a in oyunlar],
    "json_olumlu": [oyunlar[a]["positive"] for a in oyunlar],
    "json_olumsuz": [oyunlar[a]["negative"] for a in oyunlar],
}, index=list(oyunlar))

oy_karsilastirma["csv_olumlu"] = dosya_ozeti["recommended"]
oy_karsilastirma["csv_olumsuz"] = dosya_ozeti["not_recommended"]

oy_karsilastirma = oy_karsilastirma.fillna({"csv_olumlu": 0, "csv_olumsuz": 0})
oy_karsilastirma["json_oy"] = oy_karsilastirma["json_olumlu"] + oy_karsilastirma["json_olumsuz"]
oy_karsilastirma["csv_oy"] = oy_karsilastirma["csv_olumlu"] + oy_karsilastirma["csv_olumsuz"]

oy_karsilastirma.head()

,ad,json_olumlu,json_olumsuz,csv_olumlu,csv_olumsuz,json_oy,csv_oy
1724540,Welcome to Goodland,25,8,25.0,8.0,33,33.0
2201840,그녀가 다시 노래할 때까지.,8,2,0.0,0.0,10,0.0
1364180,Counter Operation Online,1,1,0.0,0.0,2,0.0
2218750,Halls of Torment,24098,1044,9898.0,922.0,25142,10820.0
3262620,剑武玄幻之旅,8,2,0.0,0.0,10,0.0


In [6]:
sadece_json = (oy_karsilastirma["json_oy"] > 0) & (oy_karsilastirma["csv_oy"] == 0)
sadece_csv = (oy_karsilastirma["json_oy"] == 0) & (oy_karsilastirma["csv_oy"] > 0)
ikisinde_de = (oy_karsilastirma["json_oy"] > 0) & (oy_karsilastirma["csv_oy"] > 0)
hicbirinde = (oy_karsilastirma["json_oy"] == 0) & (oy_karsilastirma["csv_oy"] == 0)

print("Sadece games.json'da oy var:", sadece_json.sum())
print("Sadece CSV'de oy var:", sadece_csv.sum())
print("İkisinde de oy var:", ikisinde_de.sum())
print("Hiçbirinde oy yok:", hicbirinde.sum())
print("Toplam:", sadece_json.sum() + sadece_csv.sum() + ikisinde_de.sum() + hicbirinde.sum())

Sadece games.json'da oy var: 27595
Sadece CSV'de oy var: 6068
İkisinde de oy var: 17039
Hiçbirinde oy yok: 14984
Toplam: 65686


**Bulgu:** 65.686 oyun dört gruba ayrılıyor: sadece games.json'da oy 27.595 · sadece CSV'de oy 6.068 · ikisinde de 17.039 · hiçbirinde 14.984.
İlk iki grupta oran kaynağı zorunlu olarak belli. Karar, ikisinde de oy olan 17.039 oyun için.
Örnek: Halls of Torment, games.json 25.142 oy (%95,8 olumlu), CSV 10.820 yorum (%91,5). CSV eksik toplanmış.
Not: CSV sütunları NaN yüzünden ondalığa döndü (25.0); temizlikte int'e çevrilecek.

In [7]:
ikisi = oy_karsilastirma[ikisinde_de].copy()

ikisi["json_oran"] = ikisi["json_olumlu"] / ikisi["json_oy"] * 100
ikisi["csv_oran"] = ikisi["csv_olumlu"] / ikisi["csv_oy"] * 100
ikisi["fark_puan"] = (ikisi["csv_oran"] - ikisi["json_oran"]).abs()

ikisi["secilen"] = (ikisi["json_oy"] >= ikisi["csv_oy"]).map({True: "games.json", False: "csv"})

print(ikisi["secilen"].value_counts())

secilen
csv           9211
games.json    7828
Name: count, dtype: int64


In [8]:
dilimler = pd.cut(ikisi["fark_puan"], bins=[-0.01, 1, 5, 10, 100], labels=["≤1", "1–5", "5–10", ">10"])
pd.crosstab(dilimler, ikisi["secilen"], margins=True)

secilen,csv,games.json,All
fark_puan,,,
≤1,7853,7432,15285
1–5,1200,283,1483
5–10,106,72,178
>10,52,41,93
All,9211,7828,17039


**Bulgu (ikisinde de oyu olan 17.039 oyun):**
- "Daha çok oy olan kaynak" kuralı 9.211 oyunda (%54) CSV'yi, 7.828 oyunda (%46) games.json'u seçiyor. CSV'nin sık sık daha çok yorum içermesi, iki kaynağın farklı tarihlerde toplanmış olabileceğini düşündürüyor (tahmin).
- İki kaynağın oran farkı: ≤1 puan %89,7 · 1–5 puan %8,7 · 5–10 puan %1,0 · >10 puan %0,5 (93 oyun).
- Sonuç: Oyunların %98'inde kaynak seçimi önemsiz. Karar asıl 271 oyunu (fark >5 puan) etkiliyor; bunlar tek tek incelenecek.

In [9]:
buyuk_fark = ikisi[ikisi["fark_puan"] > 10].copy()
buyuk_fark["csv_json_orani"] = buyuk_fark["csv_oy"] / buyuk_fark["json_oy"]
buyuk_fark["buyuk_oy"] = buyuk_fark[["json_oy", "csv_oy"]].max(axis=1)

sutunlar = ["ad", "json_oy", "csv_oy", "csv_json_orani", "json_oran", "csv_oran", "fark_puan", "secilen"]
print(buyuk_fark.sort_values("buyuk_oy", ascending=False)[sutunlar].round(2).to_string())

                                                             ad  json_oy    csv_oy  csv_json_orani  json_oran  csv_oran  fark_puan     secilen
1172470                                           Apex Legends™   961805   50120.0            0.05      67.72     42.64      25.08  games.json
2358720                                      Black Myth: Wukong   958121  126710.0            0.13      96.30     85.39      10.90  games.json
1938090                                           Call of Duty®   643774   10290.0            0.02      58.68     30.07      28.61  games.json
1097150                                               Fall Guys   476067   92230.0            0.19      81.27     69.39      11.89  games.json
1063730                                     New World: Aeternum   283811   82978.0            0.29      68.61     53.71      14.90  games.json
1517290                                       Battlefield™ 2042   279694   66890.0            0.24      45.17     32.54      12.63  games.json

**Bulgu (oran farkı >10 puan olan 93 oyun):**
- **Grup A, 42 popüler oyun** (Apex, CoD, Sims 4, FIFA 22...): CSV, games.json oylarının sadece %1–60'ını içeriyor ve 41 oyunda CSV oranı daha düşük. CSV'nin belirli bir dönemi kapsadığı düşünülüyor (tahmin, post_date ile kontrol edilecek). Kural games.json'u seçiyor ✔
- **Grup B, 51 küçük oyun:** games.json'da 1–100 oy, farklar küçük sayı gürültüsü. Şüpheli küme (2907xxx–2921xxx, ~101 yorum, ~%91) burada. Kural CSV'yi seçiyor ✔

**Karar 2:** Olumlu oran, hangi kaynakta daha çok oy varsa ondan hesaplanır. SQL'e `oran_kaynagi` ve `yorum_kapsami` (CSV yorum / games.json oy) sütunları eklenir.
**Açık konu (06–07):** Grup A'da SQL oranı ile Chroma yorumları çelişiyor (Apex: %68 ve %43). `yorum_kapsami` düşük oyunlarda çözüm yöntemleri denenecek.

In [10]:
def yil_ozeti(app_id):
    dosya = next(YORUM_KLASORU.glob(f"{app_id}_*.csv"))
    yorumlar = pd.read_csv(dosya, usecols=["post_date", "recommend"])
    yorumlar["tarih"] = pd.to_datetime(yorumlar["post_date"], format="%B %d, %Y", errors="coerce")
    yorumlar["olumlu"] = yorumlar["recommend"] == "Recommended"

    print(oyunlar[app_id]["name"], "| dosya:", dosya.name, "| çıkış tarihi:", oyunlar[app_id]["release_date"])
    print("games.json olumlu oran:", round(ikisi.loc[app_id, "json_oran"], 1))
    print("Tarihi okunamayan yorum:", yorumlar["tarih"].isna().sum())
    print("Okunamayan örnekler:", yorumlar.loc[yorumlar["tarih"].isna(), "post_date"].head(5).tolist())
    print("İlk yorum:", yorumlar["tarih"].min().date(), "| Son yorum:", yorumlar["tarih"].max().date())

    ozet = yorumlar.groupby(yorumlar["tarih"].dt.year).agg(
        yorum_sayisi=("olumlu", "size"),
        olumlu_yuzde=("olumlu", "mean"),
    )
    ozet["olumlu_yuzde"] = (ozet["olumlu_yuzde"] * 100).round(1)
    print(ozet.to_string())
    print()

for app_id in ["1222670", "1172470"]:
    yil_ozeti(app_id)

The Sims™ 4 | dosya: 1222670_9140.csv | çıkış tarihi: Jun 18, 2020
games.json olumlu oran: 87.4
Tarihi okunamayan yorum: 0
Okunamayan örnekler: []
İlk yorum: 2020-06-18 | Son yorum: 2024-12-02
       yorum_sayisi  olumlu_yuzde
tarih                            
2020           1180          65.1
2021           1734          67.4
2022           3055          43.0
2023           1753          58.7
2024           1418          59.3

Apex Legends™ | dosya: 1172470_50120.csv | çıkış tarihi: Nov 4, 2020
games.json olumlu oran: 67.7
Tarihi okunamayan yorum: 0
Okunamayan örnekler: []
İlk yorum: 2020-11-04 | Son yorum: 2024-12-05
       yorum_sayisi  olumlu_yuzde
tarih                            
2020           4770          84.7
2021           7855          63.5
2022           5732          54.4
2023          10170          46.1
2024          21593          21.0



### Sims 4 ve Apex: yorum tarihleri (kaynak: ham CSV, post_date)
- İki CSV de oyunun Steam çıkış gününden 2024 sonuna kadar uzanıyor, tarihi okunamayan yorum yok. "CSV belli bir dönemi içeriyor" tahmini **yanlış**.
- **Apex:** olumlu oran yıllara göre %85 → %64 → %54 → %46 → %21. Yorumların %43'ü 2024'ten. Çelişki gerçek bir zaman etkisi (games.json %68 ömür boyu birikim).
- **Sims 4:** hiçbir yıl %87'ye yaklaşmıyor (%43–67). Sebep zaman değil, yorumların seçilme biçimi (tahmin: 153 bin oydan 9 bin yorum, muhtemelen "en faydalı" sıralamasıyla toplandı).
- Sonuç: Çelişkinin iki farklı nedeni var. Yorum tarihlerini SQL'de tutmak, agent'a yıllara göre oran hesaplama yeteneği verir.
**Not:** Steam'de oy vermenin tek yolu yorum yazmak. Yani games.json positive/negative = yorum sayısı, CSV satırları = yorumların kendisi; iki kaynak aynı şeyi sayıyor.
- Sayılar yakınsa: büyük olan daha geç alınmış (arada yeni yorum gelmiş). 1–5 puan farklı oyunların çoğunda CSV'nin büyük olması bununla uyumlu.
- CSV çok küçükse (Sims 4 %6, FIFA 22 %1): zaman farkı açıklamaz, yorumların sadece bir kısmı toplanmış.

**Karar (yorum tarihleri):** Her yorumun `post_date`'i SQL'de tarih olarak tutulur; agent yıllara göre yorum sayısı ve olumlu oran hesaplayabilir.
Agent kuralları / test soruları (07–08):
1. Çıkıştan önceki yıl sorulursa → "oyun X'te çıktı" (release_date, games.json)
2. Erken erişim yorumları çıkıştan önce olabilir (early_access_review, CSV)
3. Az yorumlu yılda oranı yorum sayısıyla ver, uyar
4. Veri 2024 sonunda bitiyor
5. yorum_kapsami düşükse yıllık oranlar da yanlı olabilir

## Karar 3: Yazılım (oyun olmayan) kayıtlar
games.json'da Utilities, Photo Editing gibi oyun olmayan türler var. Önce `genres` alanındaki tüm türleri sayıyoruz.

In [11]:
tum_turler = [tur for a in oyunlar for tur in oyunlar[a]["genres"]]
tur_sayilari = pd.Series(tum_turler).value_counts()

print("Farklı tür sayısı:", len(tur_sayilari))
print(tur_sayilari.to_string())

Farklı tür sayısı: 30
Indie                    42343
Casual                   27306
Adventure                24612
Action                   24610
Simulation               12859
Strategy                 11801
RPG                      11639
Early Access              6839
Free To Play              6489
Sports                    2574
Racing                    2219
Massively Multiplayer     1435
Utilities                  481
Design & Illustration      273
Animation & Modeling       223
Game Development           199
Education                  197
Video Production           170
Audio Production           100
Photo Editing               90
Software Training           86
Web Publishing              52
Accounting                   7
Movie                        1
Documentary                  1
Episodic                     1
Short                        1
Tutorial                     1
360 Video                    1
Violent                      1


In [13]:
OYUN_TURLERI = {"Indie", "Casual", "Adventure", "Action", "Simulation", "Strategy",
                "RPG", "Sports", "Racing", "Massively Multiplayer"}
YAZILIM_TURLERI = {"Utilities", "Design & Illustration", "Animation & Modeling", "Game Development",
                   "Education", "Video Production", "Audio Production", "Photo Editing",
                   "Software Training", "Web Publishing", "Accounting",
                   "Movie", "Documentary", "Episodic", "Short", "Tutorial", "360 Video"}

def kayit_tipi(turler):
    turler = set(turler)
    oyun_var = len(turler & OYUN_TURLERI) > 0
    yazilim_var = len(turler & YAZILIM_TURLERI) > 0
    if oyun_var and yazilim_var:
        return "karisik"
    if yazilim_var:
        return "sadece_yazilim"
    if oyun_var:
        return "sadece_oyun"
    return "tur_yok"

oy_karsilastirma["kayit_tipi"] = [kayit_tipi(oyunlar[a]["genres"]) for a in oy_karsilastirma.index]
print(oy_karsilastirma["kayit_tipi"].value_counts())

kayit_tipi
sadece_oyun       59649
tur_yok            5205
sadece_yazilim      615
karisik             217
Name: count, dtype: int64


In [14]:
oy_karsilastirma["turler"] = [", ".join(oyunlar[a]["genres"]) for a in oy_karsilastirma.index]

for tip in ["sadece_yazilim", "karisik"]:
    print("=====", tip, "(en çok oylu 15) =====")
    secilen = oy_karsilastirma[oy_karsilastirma["kayit_tipi"] == tip]
    print(secilen.sort_values("json_oy", ascending=False)[["ad", "json_oy", "csv_oy", "turler"]].head(15).to_string())
    print()

===== sadece_yazilim (en çok oylu 15) =====
                                      ad  json_oy  csv_oy                                                                                turler
1787090                     MyDockFinder     4939  4993.0                                                                             Utilities
1009850            OVR Advanced Settings     4171  4220.0                                                                             Utilities
1812620                              DSX     3994  4097.0                                                                             Utilities
1486350             VRoid Studio v1.29.4     3951  3978.0                             Free To Play, Animation & Modeling, Design & Illustration
616720                    Live2DViewerEX     3379  3390.0                                                       Animation & Modeling, Utilities
1364390               Animaze by FaceRig     3074  3100.0                                   

**Bulgu (games.json genres):** sadece_oyun 59.649 · tur_yok 5.205 (4.817'si iskelet) · sadece_yazilim 615 · karisik 217.
- sadece_yazilim: en popülerleri gerçekten yazılım (MyDockFinder, DSX, VEGAS Pro, VR araçları). Çoğunun yorum CSV'si de var.
- karisik: çoğu yine yazılım (Crosshair X 9 türle "hepsini işaretle" spam'i, VTube Studio, RPG Maker MZ). Gri bölge: masaüstü evcil hayvanlar, Pixel Art Academy.
- Sonuç: genres alanını geliştirici seçiyor ve karışık grupta güvenilir değil.
**Karar 3:** sadece_yazilim (615) atılır. Karışık (217) için: ...

In [16]:
YAZILIM_ETIKETLERI = {"Software", "Utilities", "Design & Illustration", "Animation & Modeling",
                      "Video Production", "Audio Production", "Game Development", "Photo Editing",
                      "Software Training", "Web Publishing", "Accounting"}

def ilk_etiketler(app_id, n=5):
    etiketler = oyunlar[app_id]["tags"]
    if not isinstance(etiketler, dict):
        return []
    return sorted(etiketler, key=etiketler.get, reverse=True)[:n]

karisik = oy_karsilastirma[oy_karsilastirma["kayit_tipi"] == "karisik"].copy()
karisik["ilk5_etiket"] = [ilk_etiketler(a) for a in karisik.index]
karisik["yazilim_etiketi_var"] = [len(set(e) & YAZILIM_ETIKETLERI) > 0 for e in karisik["ilk5_etiket"]]

print(karisik["yazilim_etiketi_var"].value_counts())
print()
for deger in [True, False]:
    print("===== ilk 5 etikette yazılım etiketi var:", deger, "=====")
    grup = karisik[karisik["yazilim_etiketi_var"] == deger].sort_values("json_oy", ascending=False)
    print(grup[["ad", "json_oy", "ilk5_etiket"]].to_string())
    print()

yazilim_etiketi_var
True     123
False     94
Name: count, dtype: int64

===== ilk 5 etikette yazılım etiketi var: True =====
                                                             ad  json_oy                                                                                                   ilk5_etiket
1366800                                             Crosshair X    21965                                                                [Action, Indie, Adventure, Utilities, Shooter]
1325860                                            VTube Studio     4974                                                            [Indie, Mature, Anime, Animation & Modeling, NSFW]
1477830                            HudSight - crosshair overlay     4146                                           [Utilities, Video Production, Action, Software Training, Education]
1634680                                            Ficket: 追剧引擎     1917                                                               [Indie,

**Bulgu (karışık 217, oyuncu etiketleri `tags`):** İlk 5 etikette yazılım etiketi olan 123 kaydın çoğu gerçekten yazılım (Crosshair X, RPG Maker MZ, VTube Studio); ~5–10 az oylu oyun yanlış yakalanıyor (Pixel Art Academy, Neighborhood). Yazılım etiketi olmayan 94 kaydın 34'ü etiketli ve çoğu oyun; 60'ı etiketsiz ve hepsi 0 oylu.
Not: `tags` etiketsiz oyunlarda dict değil boş liste `[]`. Karar 5'te düzeltilecek.
**Karar 3:** Atılan: sadece_yazilim (615) + karışık ve ilk 5 etikette yazılım etiketi olan (123) = 738 kayıt. Kalan: 64.948.

In [17]:
oy_karsilastirma["yazilim"] = oy_karsilastirma["kayit_tipi"] == "sadece_yazilim"
yazilim_etiketli_karisik = karisik.index[karisik["yazilim_etiketi_var"]]
oy_karsilastirma.loc[yazilim_etiketli_karisik, "yazilim"] = True

print("Yazılım olarak işaretlenen:", oy_karsilastirma["yazilim"].sum())

Yazılım olarak işaretlenen: 738


In [18]:
sadece_oyun = oy_karsilastirma[oy_karsilastirma["kayit_tipi"] == "sadece_oyun"].copy()
sadece_oyun["ilk3_etiket"] = [ilk_etiketler(a, n=3) for a in sadece_oyun.index]
sadece_oyun["software_ilk3te"] = [("Software" in e) or ("Utilities" in e) for e in sadece_oyun["ilk3_etiket"]]

print("İlk 3 etikette Software/Utilities olan:", sadece_oyun["software_ilk3te"].sum())
kacak = sadece_oyun[sadece_oyun["software_ilk3te"]].sort_values("json_oy", ascending=False)
print(kacak[["ad", "json_oy", "csv_oy", "turler", "ilk3_etiket"]].head(40).to_string())

İlk 3 etikette Software/Utilities olan: 80
                                                      ad  json_oy  csv_oy                                                               turler                                      ilk3_etiket
2113850                       Spirit City: Lofi Sessions     6619  8269.0                                            Casual, Indie, Simulation                      [Cozy, Relaxing, Utilities]
1588530                                Dungeon Alchemist     2961  2995.0                                                    RPG, Early Access  [Software, Level Editor, Procedural Generation]
2826180                                      Chill Pulse     1842  1892.0                                 Adventure, Casual, Indie, Simulation                  [Casual, Simulation, Utilities]
2498570                                  Canvas of Kings      376   392.0                                             Indie, RPG, Early Access     [Software, Level Editor, Dungeons & Dragon

**Ek kontrol (sadece_oyun, ilk 3 oyuncu etiketinde Software/Utilities):** 80 kayıt. Yaklaşık yarısı gerçek yazılım (Dungeon Alchemist, Logitech VR Ink Driver), yarısı oyun ya da şaka etiketli (Paint Drying Simulator, Mystery Box 5). Az oylu oyunlarda etiketler de güvenilmez.
**Karar:** Hepsi tutuldu.

## Karar 4: Açıklama alanı (Chroma için)
games.json'da üç açıklama alanı var: short_description, about_the_game, detailed_description. Boş olanları, uzunlukları ve tekrarı ölçüyoruz.

In [19]:
alanlar = ["short_description", "about_the_game", "detailed_description"]
uzunluk = pd.DataFrame(
    {alan: [len(oyunlar[a][alan] or "") for a in oyunlar] for alan in alanlar},
    index=list(oyunlar),
)

print("Boş olan kayıt sayısı:")
print((uzunluk == 0).sum())
print()
print(uzunluk.describe().round(0).to_string())
print()

dolu = [a for a in oyunlar if oyunlar[a]["about_the_game"]]
icinde = sum(oyunlar[a]["about_the_game"] in oyunlar[a]["detailed_description"] for a in dolu)
print("about_the_game dolu olan:", len(dolu), "| detailed_description içinde aynen geçen:", icinde)

Boş olan kayıt sayısı:
short_description       5052
about_the_game          5119
detailed_description    5097
dtype: int64

       short_description  about_the_game  detailed_description
count            65686.0         65686.0               65686.0
mean               178.0          1180.0                1198.0
std                 90.0          1274.0                1294.0
min                  0.0             0.0                   0.0
25%                113.0           564.0                 569.0
50%                194.0           961.0                 970.0
75%                256.0          1544.0                1565.0
max                342.0         95482.0               95482.0

about_the_game dolu olan: 60567 | detailed_description içinde aynen geçen: 60529


In [20]:
ornek_id = "2218750"
print("OYUN:", oyunlar[ornek_id]["name"])
for alan in alanlar:
    print("\n=====", alan, "=====")
    print(oyunlar[ornek_id][alan])

OYUN: Halls of Torment

===== short_description =====
Slay hordes of terrifying monsters in this horde survival roguelite. Descend into the Halls of Torment where the Lords of the underworld await you. Treasures, magical trinkets, and a growing cast of heroes will grant you the power to vanquish these horrors from beyond. How long can you survive?

===== about_the_game =====
Halls of Torment is a horde survival game with a pre-rendered retro look reminiscent of RPGs from the late 90s. Choose one of many hero characters and descend into the deadly Halls of Torment. Fight unholy horrors from beyond and survive wave after wave of enemies until you face one of the tormented Lords. Strengthen your hero with character traits, abilities, and items. Create a new powerful build during each run. Explore various underground expanses and find new powerful items that enable you to venture even deeper into the abyss. Features Quick and casual 30 minute runs Old school pre-rendered art style Quest-ba

**Bulgu (games.json açıklama alanları):**
- Üç alanda da ~5.100 boş kayıt (4.817'si iskelet).
- short_description en fazla 342 karakter (medyan 194). about_the_game medyan 961, ama en uzunu 95.482 karakter → embedding için kesilmeli.
- about_the_game, 60.567 oyunun 60.529'unda (%99,94) detailed_description içinde aynen geçiyor → detailed_description tekrar.
- Metinlerde HTML bozulması var (`&amp;amp;` = `&`) → html.unescape ile düzeltilecek.
**Karar 4:** short + about birlikte, about belli bir uzunlukta kesilir: önce özet, sonra ayrıntı.

In [21]:
about_uzunluk = uzunluk["about_the_game"]

for sinir in [2000, 3000, 5000, 10000, 20000]:
    print(f"{sinir:>6} karakterden uzun: {(about_uzunluk > sinir).sum():>6} oyun")
print()

en_uzunlar = about_uzunluk.sort_values(ascending=False).head(10)
for app_id, uz in en_uzunlar.items():
    print(f"===== {oyunlar[app_id]['name']} | {uz} karakter | json oy: {oy_karsilastirma.loc[app_id, 'json_oy']} =====")
    print(oyunlar[app_id]["about_the_game"][:500])
    print()

  2000 karakterden uzun:   9188 oyun
  3000 karakterden uzun:   2764 oyun
  5000 karakterden uzun:    461 oyun
 10000 karakterden uzun:     53 oyun
 20000 karakterden uzun:     14 oyun

===== Crazy Zombies | 95482 karakter | json oy: 4 =====
Looking for a bit of leisure and relaxation in the midst of busy life? Crazy Zombies is just such a casual puzzle game tailored for you. Built by independent developers, it combines the strategic fun of survival challenges in fighting zombies while maintaining extremely low requirements for computer configuration, ensuring that every player can easily enjoy the fun brought by the game. The game takes you into a world ruled by zombies. As one of the survivors, you will embark on an adventure full

===== Aircraft Carrier Survival: Tutorial | 67521 karakter | json oy: 1183 =====
[, , , ,Take control of the aircraft carrier, its crew, officers and planes during WW2. Give an example of military prowess by choosing the right equipment, crew, and escort f

**Bulgu (uzun açıklamalar):** about_the_game 2.000 karakterden uzun 9.188 oyun, 5.000'den uzun 461, 20.000'den uzun 14.
En uzun 10 açıklama neredeyse tamamen çöp: 0 oylu Çin yayıncı spam'i (QQ/WeChat numaraları, hediye kodları) ve **anahtar kelime doldurma** ("其他游戏推荐: 最终幻想7, 赛博朋克2077..." gibi ünlü oyun adı listeleri). Bir oyunda (Aircraft Carrier Survival: Tutorial) bozuk metin parçaları (`[, , ,`).
**Risk:** Anahtar kelime doldurulmuş açıklamalar anlamsal aramada alakasız eşleşme üretir ("Cyberpunk 2077'ye benzer oyun" → 0 oylu spam oyun).
**Sonuç:** Uzunluk bilgi değil, çoğunlukla spam/bozukluk göstergesi. Açıklama ~2.000 karakterde kesilecek (kesin değer embedding adımında); spam kalıpları ayrıca temizlenecek.

In [22]:
baslangic = pd.Series(
    [oyunlar[a]["about_the_game"][:40] for a in oyunlar if oyunlar[a]["about_the_game"]]
)
print(baslangic.value_counts().head(30).to_string())

Help the beautiful girls to immerse them    107
A hidden object game where you must find     60
Find the objects that are hidden on the      54
Collect a 3D puzzle, transferring things     46
(This game, as well as many others in ou     24
Solve the puzzles to reveal and collect      21
Train your brain with this memory game,      20
Hentai is not just 2D girls, this is a w     18
----------------------------------------     18
※All characters involved in sexual conte     17
About the game An explicit erotic visual     17
Gameplay: Move All Parts to Form the Com     16
*** This game is only available in Chine     14
An erotic VISUAL NOVEL featuring HD grap     14
In 2082, the human species lives in peac     11
Search an incredibly detailed hand-drawn     10
Join the Discord We have now opened the      10
GamePlay Enjoy the story while completin     10
GamePlay Enjoy a sensual story while com     10
Pick a classic card game, any classic ca      9
10mg is a collection of 10 minute long g

In [23]:
isaretler = {
    "其他游戏推荐 (oyun adı doldurma)": "其他游戏推荐",
    "QQ / qq": "qq",
    "微信 (WeChat)": "微信",
    "Discord": "discord",
    "Wishlist": "wishlist",
    "redeem / code": "redeem",
    "Mature Content": "mature content",
    "[, , (bozuk liste)": "[, ,",
}

aciklamalar = {a: oyunlar[a]["about_the_game"].lower() for a in oyunlar if oyunlar[a]["about_the_game"]}
for etiket, kelime in isaretler.items():
    gecen = [a for a, metin in aciklamalar.items() if kelime in metin]
    oylu = sum(oy_karsilastirma.loc[a, "json_oy"] >= 100 for a in gecen)
    print(f"{etiket:<35} {len(gecen):>6} oyun | bunların 100+ oylu olanı: {oylu}")

其他游戏推荐 (oyun adı doldurma)               9 oyun | bunların 100+ oylu olanı: 0
QQ / qq                                418 oyun | bunların 100+ oylu olanı: 85
微信 (WeChat)                             38 oyun | bunların 100+ oylu olanı: 8
Discord                               1493 oyun | bunların 100+ oylu olanı: 363
Wishlist                               458 oyun | bunların 100+ oylu olanı: 82
redeem / code                           91 oyun | bunların 100+ oylu olanı: 23
Mature Content                          80 oyun | bunların 100+ oylu olanı: 7
[, , (bozuk liste)                       1 oyun | bunların 100+ oylu olanı: 1


**Bulgu (kalıp metinler, about_the_game ilk 40 karakter):** En sık başlangıçlar seri üretim oyunlardan ("Help the beautiful girls..." 107 oyun, gizli nesne ve yapboz serileri) ve yetişkin içerik uyarılarından geliyor. Risk: anlamsal aramada aynı seriden çok sayıda neredeyse aynı sonuç (06'da çeşitlendirme).
**Şüpheli kelimeler:** Anahtar kelime doldurma ("其他游戏推荐") sadece 9 oyunda (hepsi <100 oy) → başlıktan sonrası kesilecek. QQ, Discord, Wishlist yaygın ve popüler oyunlarda da var → dokunulmayacak.
**Yeni karar:** Yetişkin içerik nasıl ele alınacak? (tut / işaretle / at)

In [24]:
YETISKIN_ETIKETLERI = {"Sexual Content", "Nudity", "NSFW", "Hentai"}

oy_karsilastirma["yetiskin_herhangi"] = [
    len(set(ilk_etiketler(a, n=20)) & YETISKIN_ETIKETLERI) > 0 for a in oy_karsilastirma.index
]
oy_karsilastirma["yetiskin_ilk5"] = [
    len(set(ilk_etiketler(a, n=5)) & YETISKIN_ETIKETLERI) > 0 for a in oy_karsilastirma.index
]

print("Herhangi bir sırada yetişkin etiketi:", oy_karsilastirma["yetiskin_herhangi"].sum())
print("İlk 5'te yetişkin etiketi:", oy_karsilastirma["yetiskin_ilk5"].sum())
print()

sadece_herhangi = oy_karsilastirma[oy_karsilastirma["yetiskin_herhangi"] & ~oy_karsilastirma["yetiskin_ilk5"]]
print("Sadece 'herhangi' kuralının yakaladığı:", len(sadece_herhangi))
gosterilecek = sadece_herhangi.sort_values("json_oy", ascending=False).head(25)
for a in gosterilecek.index:
    etiketler = oyunlar[a]["tags"]
    yetiskin_oy = {e: etiketler[e] for e in YETISKIN_ETIKETLERI if e in etiketler}
    print(f"{oyunlar[a]['name'][:40]:<40} | json oy: {gosterilecek.loc[a, 'json_oy']:>7} | {yetiskin_oy}")

Herhangi bir sırada yetişkin etiketi: 2108
İlk 5'te yetişkin etiketi: 927

Sadece 'herhangi' kuralının yakaladığı: 1181
Baldur's Gate 3                          | json oy:  669941 | {'Nudity': 1403, 'Sexual Content': 1942}
New World: Aeternum                      | json oy:  283811 | {'Sexual Content': 121}
Battlefield™ V                           | json oy:  231618 | {'Sexual Content': 212}
Lost Ark                                 | json oy:  200946 | {'Sexual Content': 201}
Borderlands 3                            | json oy:  139505 | {'Nudity': 139}
Helltaker                                | json oy:  121325 | {'Hentai': 312, 'Sexual Content': 668}
FIFA 22                                  | json oy:  121164 | {'Sexual Content': 58}
Resident Evil 3                          | json oy:   85845 | {'Sexual Content': 119}
Persona 5 Royal                          | json oy:   85183 | {'Sexual Content': 376}
Mafia: Definitive Edition                | json oy:   80041 | {'Sexual Content': 21

In [25]:
ilk5_yetiskin = oy_karsilastirma[oy_karsilastirma["yetiskin_ilk5"]].sort_values("json_oy", ascending=False).head(25)
for a in ilk5_yetiskin.index:
    print(f"{oyunlar[a]['name'][:40]:<40} | json oy: {ilk5_yetiskin.loc[a, 'json_oy']:>7} | ilk 5: {ilk_etiketler(a)}")

Cyberpunk 2077                           | json oy:  781131 | ilk 5: ['Cyberpunk', 'Open World', 'Nudity', 'RPG', 'Singleplayer']
Resident Evil Village                    | json oy:  112368 | ilk 5: ['Survival Horror', 'Horror', 'First-Person', 'Singleplayer', 'Sexual Content']
DEVOUR                                   | json oy:   79359 | ilk 5: ['Horror', 'Online Co-Op', 'Multiplayer', 'Survival Horror', 'Sexual Content']
Love Is All Around                       | json oy:   44832 | ilk 5: ['Dating Sim', 'Singleplayer', 'Sexual Content', 'FMV', 'Visual Novel']
NEEDY STREAMER OVERLOAD                  | json oy:   41509 | ilk 5: ['Psychological Horror', 'Anime', 'Sexual Content', 'Visual Novel', 'Cute']
Bright Memory: Infinite                  | json oy:   37948 | ilk 5: ['Action', 'Female Protagonist', 'Sexual Content', 'FPS', 'Indie']
嗜血印 Bloody Spell                         | json oy:   34140 | ilk 5: ['Nudity', 'Female Protagonist', 'Sexual Content', 'Action RPG', 'Hack and Slash']

In [26]:
oy_karsilastirma["yetiskin_sayisi"] = [
    len(set(ilk_etiketler(a, n=5)) & YETISKIN_ETIKETLERI) for a in oy_karsilastirma.index
]
print(oy_karsilastirma["yetiskin_sayisi"].value_counts().sort_index())
print()

iki_ve_ustu = oy_karsilastirma[oy_karsilastirma["yetiskin_sayisi"] >= 2].sort_values("json_oy", ascending=False).head(25)
for a in iki_ve_ustu.index:
    print(f"{oyunlar[a]['name'][:40]:<40} | json oy: {iki_ve_ustu.loc[a, 'json_oy']:>7} | ilk 5: {ilk_etiketler(a)}")

yetiskin_sayisi
0    64759
1      391
2      322
3      154
4       60
Name: count, dtype: int64

嗜血印 Bloody Spell                         | json oy:   34140 | ilk 5: ['Nudity', 'Female Protagonist', 'Sexual Content', 'Action RPG', 'Hack and Slash']
Summer Memories                          | json oy:   19457 | ilk 5: ['Sexual Content', 'Nudity', 'Mature', 'Hentai', 'Pixel Graphics']
Action Taimanin                          | json oy:   18044 | ilk 5: ['Sexual Content', 'Nudity', 'Hentai', 'Mature', 'Free to Play']
House Party                              | json oy:   14612 | ilk 5: ['Nudity', 'Mature', 'NSFW', 'Simulation', 'Choices Matter']
She Will Punish Them                     | json oy:   12384 | ilk 5: ['Sexual Content', 'Nudity', 'Mature', 'Female Protagonist', '3D']
VillageRhapsody                          | json oy:   11569 | ilk 5: ['Sexual Content', 'Nudity', 'Mature', 'Hentai', 'RPG']
Soulworker                               | json oy:   10448 | ilk 5: ['Anime', 'Free to P

In [27]:
oy_karsilastirma["yetiskin"] = oy_karsilastirma["yetiskin_sayisi"] >= 2
oy_karsilastirma = oy_karsilastirma.drop(columns=["yetiskin_herhangi", "yetiskin_ilk5"])

print("Yetişkin olarak işaretlenen:", oy_karsilastirma["yetiskin"].sum())

Yetişkin olarak işaretlenen: 536


**Bulgu:** İlk 5 kuralı da ana akım oyunları yakalıyor (Cyberpunk 2077'de "Nudity" 3. sırada, RE Village, DEVOUR). Fark: ana akım oyunlarda ilk 5'te 1 yetişkin etiketi var, yetişkin odaklı oyunlarda 2–4.
İlk 5'teki yetişkin etiketi sayısı: 0 → 64.759 · 1 → 391 · 2 → 322 · 3 → 154 · 4 → 60.
**Karar (yetişkin içerik):** Oyunlar tutulur. `yetiskin` = ilk 5 oyuncu etiketinde ≥2 yetişkin etiketi (536 oyun) → agent varsayılan olarak önermez. `yetiskin_etiket_sayisi` (0–4) de tutulur → çocuklara yönelik sorularda 1 olanlar da elenebilir.
Test soruları: "korku oyunu öner" (RE Village gelmeli) · "çocuğuma oyun öner" (gelmemeli).

## Karar 5: Oyuncu etiketleri (tags)
Her oyunda en fazla 20 etiket ve oy sayıları var. Tip sorunu: etiketli oyunda dict, etiketsiz oyunda boş liste `[]`.
Ölçüm: tip dağılımı, oyun başına etiket sayısı, farklı etiket sayısı, en yaygın etiketler.

In [28]:
tip_sayilari = pd.Series([type(oyunlar[a]["tags"]).__name__ for a in oyunlar]).value_counts()
print("tags alanının tipi:")
print(tip_sayilari.to_string())
print()

etiket_sayisi = pd.Series([len(oyunlar[a]["tags"]) for a in oyunlar])
print("Oyun başına etiket sayısı:")
print(etiket_sayisi.value_counts().sort_index().to_string())
print()

tum_etiketler = pd.Series([e for a in oyunlar if isinstance(oyunlar[a]["tags"], dict) for e in oyunlar[a]["tags"]])
print("Farklı etiket sayısı:", tum_etiketler.nunique())
print()
print("En yaygın 30 etiket (kaç oyunda geçiyor):")
print(tum_etiketler.value_counts().head(30).to_string())

tags alanının tipi:
dict    44786
list    20900

Oyun başına etiket sayısı:
0     20900
1        50
2        78
3       118
4       131
5      1051
6       935
7      1107
8      1122
9      1157
10     1256
11     1388
12     1457
13     1557
14     1676
15     1728
16     1840
17     1995
18     1956
19     1845
20    22339

Farklı etiket sayısı: 450

En yaygın 30 etiket (kaç oyunda geçiyor):
Singleplayer        33046
Indie               23128
Casual              19805
Action              19478
Adventure           18986
2D                  17472
3D                  13704
Colorful            10347
Atmospheric         10173
Puzzle               9780
Simulation           9624
Strategy             9435
Exploration          9174
Pixel Graphics       8954
RPG                  8765
Cute                 8592
Story Rich           8277
First-Person         8075
Fantasy              7558
Arcade               7481
Action-Adventure     7180
Early Access         6375
Relaxing             6200
Funn

**Bulgu (tags):** 44.786 oyunda dict, 20.900 oyunda boş liste (= etiketsiz). 22.339 oyunda 20 etiket (Steam üst sınırı). 450 farklı etiket. En yaygınlar genel (Singleplayer 33.046, Indie, Casual), ayırt edici bilgi seyrek etiketlerde.
**Karar 5:**
- SQL: ayrı tablo `oyun_etiket(app_id, etiket, oy, sira)`; türler için de `oyun_tur(app_id, tur)`. Boş liste = etiket yok.
- Chroma oyun metni: en çok oylanan ilk 10 etiket eklenir (sayı embedding testinde ayarlanabilir).

## Karar 6: Tip düzeltmeleri
games.json'daki 21 alanın Python tipleri ölçülüyor. Amaç: SQL'de doğru tiplerle (tarih, sayı, metin) saklamak.

In [29]:
ornek_id = "2218750"
for alan in oyunlar[ornek_id]:
    tipler = pd.Series([type(oyunlar[a][alan]).__name__ for a in oyunlar]).value_counts()
    ornek = repr(oyunlar[ornek_id][alan])[:70]
    print(f"{alan:<26} {dict(tipler)}")
    print(f"{'':<26} örnek: {ornek}")

name                       {'str': np.int64(65686)}
                           örnek: 'Halls of Torment'
release_date               {'str': np.int64(65686)}
                           örnek: 'Sep 24, 2024'
required_age               {'int': np.int64(65686)}
                           örnek: 0
price                      {'float': np.int64(65686)}
                           örnek: 6.66
detailed_description       {'str': np.int64(65686)}
                           örnek: 'Halls of Torment is a horde survival game with a pre-rendered retro l
about_the_game             {'str': np.int64(65686)}
                           örnek: 'Halls of Torment is a horde survival game with a pre-rendered retro l
short_description          {'str': np.int64(65686)}
                           örnek: 'Slay hordes of terrifying monsters in this horde survival roguelite. 
supported_languages        {'list': np.int64(65686)}
                           örnek: ['English', 'French', 'Italian', 'German', 'Spanish - S

**Bulgu (21 alanın tipleri):** 19 alan tek tip. Sorunlar:
- Metin → başka tip: release_date ("Sep 24, 2024" → tarih), estimated_owners ("500000 - 1000000" → alt/üst sayı), discount (52.165 str + 13.521 int → int).
- 0 = veri yok: playtime_forever alanları (%95 sıfır) → NULL. *_2weeks anlık → atılacak. peak_ccu anlık değer.
- Listeler: genres, tags, categories, supported_languages, full_audio_languages → ayrı SQL tabloları (dil tablosu Türkçe destek soruları için önemli).
- required_age: yetişkin/çocuk kararlarından sonra atmadan önce incelenecek.
- release_date tam sürüm tarihi (Halls of Torment 2024, ama 2023'ten beri erken erişimde).

In [30]:
tarih_metni = pd.Series({a: oyunlar[a]["release_date"] for a in oyunlar})
tarih = pd.to_datetime(tarih_metni, format="%b %d, %Y", errors="coerce")

print("Okunamayan tarih:", tarih.isna().sum())
print(tarih_metni[tarih.isna()].value_counts().head(30).to_string())
print()
print("Yıllara göre oyun sayısı:")
print(tarih.dt.year.value_counts().sort_index().to_string())

Okunamayan tarih: 0
Series([], )

Yıllara göre oyun sayısı:
2020     8906
2021    11241
2022    12551
2023    15113
2024    17875


**Bulgu (release_date):** 65.686 tarihin hepsi "%b %d, %Y" biçiminde, okunamayan 0. Hepsi 2020–2024 arası (2020: 8.906 → 2024: 17.875).
**Sonuç:** Veri seti sadece 2020–2024'te Steam'e çıkan oyunları içeriyor; daha eski oyunlar (Witcher 3, GTA V) yok.
**Karar:** release_date → tarih tipi (tam sürüm tarihi). Agent kuralı + test sorusu: veri setinde olmayan eski bir oyun sorulursa uydurmadan "kapsam dışı" demeli.

In [31]:
aranacak = ["Witcher", "Grand Theft Auto", "Counter-Strike", "Terraria", "Stardew", "Skyrim", "Portal", "Dota"]
for kelime in aranacak:
    bulunan = [(a, oyunlar[a]["name"], oyunlar[a]["release_date"])
               for a in oyunlar if kelime.lower() in oyunlar[a]["name"].lower()]
    print(f"{kelime} → {len(bulunan)} kayıt")
    for b in bulunan[:10]:
        print("    ", b)
print()

print("En küçük app_id'li 15 oyun:")
for a in sorted(oyunlar, key=int)[:15]:
    print(f"{a:>8} | {oyunlar[a]['name'][:45]:<45} | {oyunlar[a]['release_date']}")

Witcher → 5 kayıt
     ('2889630', 'The Witcher 3 REDkit Playtest', 'Apr 16, 2024')
     ('2684660', 'The Witcher 3 REDkit', 'May 21, 2024')
     ('1594560', 'Chrome Switcher', 'Jun 11, 2021')
     ('2376280', 'Whisker Witchery', 'Apr 26, 2023')
     ('1284410', 'GWENT: The Witcher Card Game', 'May 19, 2020')
Grand Theft Auto → 4 kayıt
     ('1546990', 'Grand Theft Auto: Vice City – The Definitive Edition', 'Jan 19, 2023')
     ('1546970', 'Grand Theft Auto III – The Definitive Edition', 'Jan 19, 2023')
     ('12210', 'Grand Theft Auto IV: The Complete Edition', 'Mar 24, 2020')
     ('1547000', 'Grand Theft Auto: San Andreas – The Definitive Edition', 'Jan 19, 2023')
Counter-Strike → 0 kayıt
Terraria → 0 kayıt
Stardew → 0 kayıt
Skyrim → 0 kayıt
Portal → 38 kayıt
     ('1462940', 'Ecto Portal', 'Jan 16, 2021')
     ('2444350', 'Enchanted Portals', 'Sep 5, 2023')
     ('1556250', 'Tezz: Portals', 'Mar 18, 2021')
     ('1917370', 'HyperPortals', 'Mar 17, 2022')
     ('2360070', 'Spaceship

In [32]:
yil_ozeti("239140")

Dying Light | dosya: 239140_179837.csv | çıkış tarihi: Jul 15, 2021
games.json olumlu oran: 95.2
Tarihi okunamayan yorum: 0
Okunamayan örnekler: []
İlk yorum: 2015-01-27 | Son yorum: 2024-12-03
       yorum_sayisi  olumlu_yuzde
tarih                            
2015           9292          91.0
2016           6660          90.6
2017           8154          89.4
2018          11695          86.1
2019          14838          90.0
2020          20770          89.5
2021          19143          87.3
2022          13468          83.8
2023          30700          94.1
2024          45117          95.1



**Test (Kaan'ın itirazı):** release_date hepsi 2020–2024 olsa da eski oyunlar var: Borderlands GOTY (app_id 8980, 2009) → 2023, GTA IV (12210) → 2020, Dying Light (2015) → 2021, 7 Days to Die → 2024 (1.0 sürümü). Witcher 3, CS, Terraria, Skyrim yok.
**Sonuç:** release_date ilk çıkış değil, Steam'deki güncel çıkış tarihi (erken erişimden çıkış, yeniden yayın). Veri seti bu tarihe göre filtrelenmiş.
**Karar:** SQL'de adı `steam_cikis_tarihi`. Agent "oyun X yılında çıktı" demez, "Steam'deki kayıtlı çıkış tarihi" der. Test soruları: "Dying Light ne zaman çıktı?", "Witcher 3 nasıl?"

**Doğrulama (Dying Light, ham CSV):** İlk yorum 2015-01-27 (gerçek çıkış), Steam tarihi (games.json) 2021-07-15. CSV'ler oyunun tüm geçmişini içeriyor; "2020–2024" sadece Steam'deki güncel çıkış tarihi için geçerli.
- `ilk_yorum_tarihi` (CSV) gerçek çıkışa yakın bir ipucu → steam_cikis_tarihi ile çok farklıysa agent bunu açıklar.
- Dying Light yıllık olumlu oran: 2015–2022 %84–91 → 2023–2024 %94–95, yorum sayısı 2024'te en yüksek (45 bin). Apex'in tersi bir hikâye.

**Geriye dönük kontrol:** CSV'lerin tüm geçmişi içerdiği ve release_date'in ilk çıkış olmadığı öğrenilince eski kararlar gözden geçirildi. Karar 1–4, oran kaynağı, Sims 4/Apex analizi ve yetişkin kararı etkilenmiyor (games.json oyları ve CSV ikisi de tüm geçmişi sayıyor).
**Düzeltilen agent kuralı:** Oyunun bir yılda var olup olmadığı `ilk_yorum_tarihi`ne (CSV) göre değerlendirilir, steam_cikis_tarihi'ne göre değil. Fark büyükse "erken erişim / yeniden yayın" diye açıklanır.
Not: Yeniden yayın aynı app_id ise yorumlar korunur (Dying Light), yeni app_id ise sıfırdan başlar (GTA Definitive Edition). Yorum temizliğinde ölçülecek.

### estimated_owners
games.json'da metin aralığı ("500000 - 1000000"). Tüm farklı aralıklar ve sayıları ölçülüyor.

In [33]:
sahip_metni = pd.Series([oyunlar[a]["estimated_owners"] for a in oyunlar])
print(sahip_metni.value_counts().to_string())

0 - 20000               41338
0 - 0                   13521
20000 - 50000            5473
50000 - 100000           2341
100000 - 200000          1376
200000 - 500000           914
500000 - 1000000          373
1000000 - 2000000         193
2000000 - 5000000         102
5000000 - 10000000         30
10000000 - 20000000        11
20000000 - 50000000         8
50000000 - 100000000        6


In [34]:
sahip_00 = pd.Series({a: oyunlar[a]["estimated_owners"] == "0 - 0" for a in oyunlar})
indirim_int = pd.Series({a: isinstance(oyunlar[a]["discount"], int) for a in oyunlar})

print(pd.crosstab(sahip_00, indirim_int, rownames=["sahip '0 - 0'"], colnames=["discount sayı mı"]))
print()

grup = oy_karsilastirma.loc[sahip_00[sahip_00].index]
print("Bu gruptaki oyunlar:", len(grup))
print("games.json'da 0 oylu:", (grup["json_oy"] == 0).sum())
print("Yorum CSV'si olan:", (grup["csv_oy"] > 0).sum())
print("peak_ccu = 0 olan:", sum(oyunlar[a]["peak_ccu"] == 0 for a in grup.index))
print()
print(grup.sort_values("csv_oy", ascending=False)[["ad", "json_oy", "csv_oy"]].head(10).to_string())

discount sayı mı  False  True 
sahip '0 - 0'                 
False             52165      0
True                  0  13521

Bu gruptaki oyunlar: 13521
games.json'da 0 oylu: 13521
Yorum CSV'si olan: 4320
peak_ccu = 0 olan: 13521

                                          ad  json_oy    csv_oy
2357570                         Overwatch® 2        0  313125.0
2073850                           THE FINALS        0  140640.0
2074920                 The First Descendant        0   91675.0
1818450                         STALCRAFT: X        0   62455.0
1180320            War of the Three Kingdoms        0   61045.0
2016590                      Dark and Darker        0   56710.0
760160   Vampire: The Masquerade - Bloodhunt        0   56057.0
1928420                          Farlight 84        0   48436.0
2923300                               Banana        0   48156.0
2139460                           Once Human        0   44510.0


**Bulgu (estimated_owners):** 13 aralık, biçim temiz. %63'ü "0 - 20000" (bilgi değeri düşük). **"0 - 0" = 13.521 oyun** ve bu oyunların hepsinde discount int, games.json oyu 0, peak_ccu 0. Aralarında Overwatch 2 (CSV'de 313 bin yorum), THE FINALS, Once Human var.
**Sonuç:** Bu 13.521 oyunda istatistik alanları doldurulmamış (muhtemelen istatistik kaynağı, ör. SteamSpy, veri vermemiş; tahmin). 01'deki "0 oylu ama CSV'li 6.068 oyun"un 4.320'si bu gruptan.
**Karar:** Bu grupta positive, negative, estimated_owners, peak_ccu, discount, playtime → NULL; `istatistik_eksik` bayrağı. `oy_sayisi` = oranın alındığı kaynaktaki oy sayısı (Overwatch 2 sıralamalardan elenmesin). estimated_owners → sahip_alt / sahip_ust, agent aralık olarak söyler.

In [35]:
ust_sinir = pd.Series({a: int(oyunlar[a]["estimated_owners"].split(" - ")[1]) for a in oyunlar})
en_buyukler = ust_sinir[ust_sinir >= 10_000_000].index
for a in sorted(en_buyukler, key=lambda x: -ust_sinir[x]):
    print(f"{oyunlar[a]['name'][:40]:<40} | {oyunlar[a]['estimated_owners']:<22} | json oy: {oy_karsilastirma.loc[a, 'json_oy']:>7} | fiyat: {oyunlar[a]['price']}")

New World: Aeternum                      | 50000000 - 100000000   | json oy:  283811 | fiyat: 59.99
Lost Ark                                 | 50000000 - 100000000   | json oy:  200946 | fiyat: 0.0
Palworld                                 | 50000000 - 100000000   | json oy:  323403 | fiyat: 29.99
Black Myth: Wukong                       | 50000000 - 100000000   | json oy:  958121 | fiyat: 59.99
Call of Duty®                            | 50000000 - 100000000   | json oy:  643774 | fiyat: 0.0
Apex Legends™                            | 50000000 - 100000000   | json oy:  961805 | fiyat: 0.0
HELLDIVERS™ 2                            | 20000000 - 50000000    | json oy:  911830 | fiyat: 39.99
ELDEN RING                               | 20000000 - 50000000    | json oy:  977477 | fiyat: 59.99
NARAKA: BLADEPOINT                       | 20000000 - 50000000    | json oy:  302586 | fiyat: 0.0
Baldur's Gate 3                          | 20000000 - 50000000    | json oy:  669941 | fiyat: 47.99
Cyberpun

**Bulgu (en çok sahibi olanlar):** Liste mantıklı (Apex, CoD, Wukong, Palworld, ELDEN RING, BG3). Overwatch 2 / THE FINALS yok (istatistik eksik grubu).
- Eski oyunlar Steam'e taşınma yoluyla da girmiş (EA: Titanfall 2, Battlefield V; Microsoft: Forza Horizon 4) → steam_cikis_tarihi bazen "Steam'e geliş tarihi".
- Fiyat tuhaflıkları: Titanfall 2 ve Battlefield 2042 4,79 $ (indirim günü olabilir), Sims 4 / FIFA 23 / CoD 0 $ (ücretsiz veya satıştan kalkmış).

In [36]:
indirim = pd.Series({a: int(oyunlar[a]["discount"]) for a in oyunlar})
fiyat = pd.Series({a: oyunlar[a]["price"] for a in oyunlar})

print("İndirimde olan oyun:", (indirim > 0).sum(), "/", len(indirim))
print(indirim[indirim > 0].describe().round(1).to_string())
print()

ornekler = {"Titanfall 2": "1237970", "Battlefield 2042": "1517290", "Sims 4": "1222670", "Halls of Torment": "2218750"}
for ad, a in ornekler.items():
    if a in oyunlar:
        print(f"{ad:<18} fiyat: {fiyat[a]:>6} | indirim: %{indirim[a]}")

İndirimde olan oyun: 3482 / 65686
count    3482.0
mean       48.6
std        23.8
min        10.0
25%        30.0
50%        50.0
75%        70.0
max        95.0

Titanfall 2        fiyat:   4.79 | indirim: %84
Battlefield 2042   fiyat:   4.79 | indirim: %92
Sims 4             fiyat:    0.0 | indirim: %0
Halls of Torment   fiyat:   6.66 | indirim: %0


**Bulgu (discount, price):** 3.482 oyun (%5) veri toplandığı gün indirimdeydi (medyan %50). price = o günün indirimli fiyatı: Titanfall 2 4,79 $ / %84 → normal 29,99 $; Battlefield 2042 4,79 $ / %92 → 59,99 $.
**Karar:** fiyat (o günkü), indirim_yuzde (int), normal_fiyat = fiyat / (1 − indirim/100). İstatistik eksik grubunda indirim_yuzde = NULL. Agent fiyat sorularında ve sıralamalarda normal_fiyat kullanır.

In [37]:
yas = pd.Series({a: oyunlar[a]["required_age"] for a in oyunlar})
print(yas.value_counts().sort_index().to_string())
print()
print(pd.crosstab(yas, oy_karsilastirma["yetiskin"]))

0     65264
3         2
5         1
7         2
10        6
12        9
13       59
15        1
16        9
17      251
18       81
20        1

yetiskin  False  True 
row_0                 
0         64729    535
3             2      0
5             1      0
7             2      0
10            6      0
12            9      0
13           59      0
15            1      0
16            9      0
17          250      1
18           81      0
20            1      0


**Bulgu (required_age):** 65.264 oyunda (%99,4) 0 = belirtilmemiş. 422 oyunda değer var (17: 251, 18: 81, 13: 59), ESRB ve PEGI karışık. Yetişkin işaretli 536 oyunun 535'inde yaş sınırı 0 → yetişkin içerik bu alanla yakalanamaz; etiket tabanlı bayrak doğru seçim.
**Karar:** yas_siniri olarak tutulur, 0 → NULL. Agent çocuk sorularında yas_siniri ≥ 17 ve yetişkin bayraklarını birlikte kullanır; NULL'u "her yaşa uygun" saymaz.

### Diller (supported_languages, full_audio_languages)
Türkçe destek soruları için kritik. Ölçüm: oyun başına dil sayısı, Türkçe destekleyen oyun sayısı, "her dili işaretleme" spam'i.

In [38]:
diller = pd.Series({a: oyunlar[a]["supported_languages"] for a in oyunlar})
dil_sayisi = diller.apply(len)

print("Farklı dil sayısı:", len({d for liste in diller for d in liste}))
print("Dil bilgisi boş olan oyun:", (dil_sayisi == 0).sum())
print("50+ dil işaretleyen:", (dil_sayisi >= 50).sum(), "| 100+ dil:", (dil_sayisi >= 100).sum())
print()

turkce_metin = diller.apply(lambda liste: "Turkish" in liste)
turkce_ses = pd.Series({a: "Turkish" in oyunlar[a]["full_audio_languages"] for a in oyunlar})
print("Türkçe metin/altyazı destekleyen:", turkce_metin.sum())
print("Türkçe seslendirmeli:", turkce_ses.sum())

Farklı dil sayısı: 103
Dil bilgisi boş olan oyun: 5124
50+ dil işaretleyen: 1188 | 100+ dil: 1160

Türkçe metin/altyazı destekleyen: 5939
Türkçe seslendirmeli: 2045


In [39]:
cok_dilli = oy_karsilastirma.loc[dil_sayisi[dil_sayisi >= 100].index]
print("100+ dil işaretleyen oyunların games.json oyu:")
print(cok_dilli["json_oy"].describe().round(0).to_string())
print()
print(cok_dilli.sort_values("json_oy", ascending=False)[["ad", "json_oy"]].head(15).to_string())

100+ dil işaretleyen oyunların games.json oyu:
count    1160.0
mean       20.0
std       170.0
min         0.0
25%         0.0
50%         0.0
75%         4.0
max      3656.0

                               ad  json_oy
1218900                    Zup! Z     3656
1216200           Coloring Game 2     3112
1239260                   Barro F     1904
1554790           Coloring Game 4     1497
1275640           Coloring Game 3     1340
1307090              Barro Racing      975
593640                   Papetura      918
1334000                    Wordle      614
1990740                  Barro GT      478
1334010                 Last Room      378
1958600                 Barro F22      370
1797610                  Barro 22      369
1262060                Battle Ram      317
1073810    Shapik: The Moon Quest      245
1227750  Puzzle - LINES AND KNOTS      241


In [40]:
dil_spam = dil_sayisi >= 100
ses_sayisi = pd.Series({a: len(oyunlar[a]["full_audio_languages"]) for a in oyunlar})

print("Seslendirmede 50+ dil işaretleyen:", (ses_sayisi >= 50).sum())
print("Türkçe metin (spam hariç):", (turkce_metin & ~dil_spam).sum())
print("Türkçe seslendirme (spam hariç):", (turkce_ses & ~dil_spam & (ses_sayisi < 50)).sum())
print()

gercek_tr = oy_karsilastirma.loc[(turkce_ses & ~dil_spam & (ses_sayisi < 50))[lambda s: s].index]
print("Türkçe seslendirmeli en popüler 15 oyun:")
print(gercek_tr.sort_values("json_oy", ascending=False)[["ad", "json_oy"]].head(15).to_string())

Seslendirmede 50+ dil işaretleyen: 916
Türkçe metin (spam hariç): 4779
Türkçe seslendirme (spam hariç): 1125

Türkçe seslendirmeli en popüler 15 oyun:
                                                 ad  json_oy
1293830                             Forza Horizon 4   260726
1326470                          Sons Of The Forest   209485
1551360                             Forza Horizon 5   197096
1144200                                Ready or Not   195584
534380   Dying Light 2 Stay Human: Reloaded Edition   173419
1238840                              Battlefield™ 1   162492
1811260                          EA SPORTS™ FIFA 23   151319
1506830                                     FIFA 22   121164
1142710                    Total War: WARHAMMER III   109550
2195250                            EA SPORTS FC™ 24   103374
1943950                        Escape the Backrooms    87214
1407200                              World of Tanks    79913
753640                                  Outer Wilds    7

**Bulgu (diller):** 103 farklı dil, 5.124 oyunda dil bilgisi boş. 100+ dil işaretleyen 1.160 oyun var (50–99 arası sadece 28) → "hepsini işaretle" spam'i. Bu oyunların medyan oyu 0, en popülerleri küçük oyunlar (Coloring Game, Barro serisi).
Türkçe: metin 5.939, seslendirme 2.045 (spam dahil, gerçek sayı daha düşük).

**Bulgu (Türkçe, spam hariç):** Türkçe metin 4.779, Türkçe seslendirme 1.125. Seslendirmede de 916 oyun 50+ dil işaretlemiş. Spam ayıklansa da şüpheli beyanlar var (Sons of the Forest, Ready or Not seslendirme listesinde).
**Sonuç:** Dil bilgisi geliştiricinin kendi beyanı, Steam doğrulamıyor.
**Karar:** Ayrı tablo `oyun_dil(app_id, dil, ses_var)` + bayraklar `dil_spam` (metinde 100+ dil) ve `ses_spam` (seslendirmede 50+ dil). Agent dil bilgisini kaynağına atfederek verir ("geliştirici belirtmiş"); spam bayraklı oyunlarda dil bilgisini kullanmaz.
Test sorusu: "Türkçe dublajlı oyun öner."
"Spam bayraklı oyunlarda agent dil bilgisini kullanmaz, kullanıcıya Steam sayfasından kendisinin kontrol etmesini önerir.”

### Kategoriler (categories)
Steam'in sabit özellik listesi (Single-player, Online Co-op, kontrolcü desteği...). "Ne destekliyor" sorularının kaynağı. Ölçüm: farklı kategori sayısı ve her birinin kaç oyunda geçtiği.

In [41]:
kategoriler = pd.Series({a: oyunlar[a]["categories"] for a in oyunlar})
print("Kategori bilgisi boş olan oyun:", (kategoriler.apply(len) == 0).sum())

tum_kategoriler = pd.Series([k for liste in kategoriler for k in liste])
print("Farklı kategori sayısı:", tum_kategoriler.nunique())
print()
print(tum_kategoriler.value_counts().to_string())

Kategori bilgisi boş olan oyun: 5312
Farklı kategori sayısı: 40

Single-player                 57384
Family Sharing                50075
Steam Achievements            27051
Steam Cloud                   14055
Full controller support       13053
Multi-player                  10205
Partial Controller Support     6958
PvP                            6676
Co-op                          5876
Online PvP                     4801
Steam Leaderboards             3925
Online Co-op                   3895
Shared/Split Screen            3765
Remote Play Together           3535
VR Only                        2537
Shared/Split Screen PvP        2480
Steam Trading Cards            2419
Tracked Controller Support     2340
Shared/Split Screen Co-op      2255
Stats                          1957
In-App Purchases               1663
Cross-Platform Multiplayer     1464
Steam Workshop                 1199
Includes level editor          1022
Remote Play on TV               782
MMO                             774

**Bulgu (categories):** 40 kategori, Steam'in sabit listesi, spam yok. 5.312 oyunda boş (çoğu iskelet). "VR Support" (18) ile "VR Supported" (533) aynı anlam. Çok oyunculu kategoriler hiyerarşik: Multi-player → PvP / Co-op → Online / LAN / Shared-Split Screen.
**Karar:** Ayrı tablo `oyun_kategori(app_id, kategori)`. "VR Support" → "VR Supported" birleştirilir. Kategori hiyerarşisi 04'te SQL aracının tablo açıklamasına yazılır (ör. "internetten co-op" = Online Co-op).
Test sorusu: "Arkadaşımla internetten oynayabileceğim co-op oyun öner."

In [42]:
sure_alanlari = ["average_playtime_forever", "median_playtime_forever", "peak_ccu"]
populer = oy_karsilastirma["json_oy"] >= 1000

for alan in sure_alanlari:
    deger = pd.Series({a: oyunlar[a][alan] for a in oyunlar})
    tum = (deger == 0).mean()
    istatistikli = (deger[~sahip_00] == 0).mean()
    pop = (deger[populer] == 0).mean()
    print(f"{alan:<26} sıfır → tümü: {tum:.1%} | istatistiği olan: {istatistikli:.1%} | 1000+ oylu: {pop:.1%}")

average_playtime_forever   sıfır → tümü: 94.7% | istatistiği olan: 93.4% | 1000+ oylu: 30.1%
median_playtime_forever    sıfır → tümü: 94.7% | istatistiği olan: 93.4% | 1000+ oylu: 30.1%
peak_ccu                   sıfır → tümü: 82.3% | istatistiği olan: 77.7% | 1000+ oylu: 2.8%


In [43]:
en_populer = oy_karsilastirma.sort_values("json_oy", ascending=False).head(10).index
for a in en_populer:
    o = oyunlar[a]
    print(f"{o['name'][:28]:<28} | ort: {o['average_playtime_forever']:>6} | medyan: {o['median_playtime_forever']:>6} | peak_ccu: {o['peak_ccu']:>7}")

ELDEN RING                   | ort:   7605 | medyan:   5671 | peak_ccu:   35146
Apex Legends™                | ort:      0 | medyan:      0 | peak_ccu:  148971
Black Myth: Wukong           | ort:      0 | medyan:      0 | peak_ccu:   86447
HELLDIVERS™ 2                | ort:      0 | medyan:      0 | peak_ccu:   33660
Cyberpunk 2077               | ort:   4893 | medyan:   3150 | peak_ccu:   39643
Phasmophobia                 | ort:   1949 | medyan:   1129 | peak_ccu:   21353
Baldur's Gate 3              | ort:   6236 | medyan:   3762 | peak_ccu:   58002
Call of Duty®                | ort:      0 | medyan:      0 | peak_ccu:  246142
Fall Guys                    | ort:      0 | medyan:      0 | peak_ccu:    1276
Valheim                      | ort:   3405 | medyan:   1506 | peak_ccu:   24855


In [44]:
istatistikli = oy_karsilastirma[~sahip_00].copy()
istatistikli["sure_sifir"] = [oyunlar[a]["average_playtime_forever"] == 0 for a in istatistikli.index]
istatistikli["ccu_sifir"] = [oyunlar[a]["peak_ccu"] == 0 for a in istatistikli.index]

basamak = pd.cut(istatistikli["json_oy"], bins=[-1, 0, 10, 100, 1000, 10000, 10**7],
                 labels=["0", "1-10", "11-100", "101-1k", "1k-10k", "10k+"])
print(istatistikli.groupby(basamak, observed=True)[["sure_sifir", "ccu_sifir"]].mean().round(3).to_string())
print()
print(basamak.value_counts().sort_index().to_string())

         sure_sifir  ccu_sifir
json_oy                       
0             0.975      0.849
1-10          1.000      0.959
11-100        0.991      0.811
101-1k        0.811      0.390
1k-10k        0.336      0.035
10k+          0.174      0.002

json_oy
0          7531
1-10      18586
11-100    16965
101-1k     6321
1k-10k     2164
10k+        598


In [45]:
pop = istatistikli[istatistikli["json_oy"] >= 1000].copy()
pop["yil"] = [int(oyunlar[a]["release_date"][-4:]) for a in pop.index]
pop["ucretsiz"] = [oyunlar[a]["price"] == 0 for a in pop.index]

print("1000+ oylu oyunlarda süre = 0 oranı, yıla göre:")
print(pop.groupby("yil")["sure_sifir"].agg(["mean", "size"]).round(3).to_string())
print()
print("Ücretsiz mi?")
print(pop.groupby("ucretsiz")["sure_sifir"].agg(["mean", "size"]).round(3).to_string())

1000+ oylu oyunlarda süre = 0 oranı, yıla göre:
       mean  size
yil              
2020  0.293   635
2021  0.287   595
2022  0.280   525
2023  0.307   550
2024  0.347   458

Ücretsiz mi?
           mean  size
ucretsiz             
False     0.291  2511
True      0.401   252


**Bulgu (oynama süresi, peak_ccu):** Birim dakika (ELDEN RING ort. 7.605 dk ≈ 127 saat). İstatistiği olan oyunlarda sıfır oranı popülerlikle düşüyor: süre %100 → %17 (10k+ oy), peak_ccu %96 → %0,2.
- peak_ccu sıfırları gerçek (o gün kimse oynamıyor).
- Süre sıfırları çoğunlukla "ölçülemedi"; popüler oyunlarda da ~%20–30 sıfır var (Apex, Wukong, CoD). Yıl etkisi yok, ücretsiz oyunlarda biraz daha sık (%40 / %29); kesin sebep veride görünmüyor.
**Karar:** ortalama_sure_saat, medyan_sure_saat (0 → NULL, dakika / 60). peak_ccu tutulur (0 gerçek; istatistik eksik grubunda NULL); agent "veri toplandığı günkü" diye belirtir.

## Karar 7: Atılan sütunlar ve SQL şeması
**Atılan:** detailed_description (kopya), *_playtime_2weeks (anlık), ham discount / genres / tags / categories / dil listeleri (temiz hâlleri ve ayrı tablolar var). Satır olarak 738 yazılım kaydı.
**oyunlar (64.948):** app_id, ad · steam_cikis_tarihi, ilk/son_yorum_tarihi · fiyat, indirim_yuzde, normal_fiyat · olumlu/olumsuz_json, olumlu/olumsuz_csv, olumlu_oran, oran_kaynagi, oy_sayisi · yorum_sayisi, yorum_var, yorum_kapsami · sahip_alt/ust, ortalama/medyan_sure_saat, peak_ccu, yas_siniri · kisa_aciklama, oyun_hakkinda · iskelet, istatistik_eksik, yetiskin, yetiskin_etiket_sayisi, dil_spam, ses_spam
**Yardımcı tablolar:** oyun_tur, oyun_etiket(oy, sira), oyun_kategori, oyun_dil(ses_var). Sonra: yorumlar.

## Uygulama: temiz oyunlar tablosu
Karar 1–7 adım adım uygulanıyor. Adım 1: tek değerli alanlar (tarih, fiyat, sahip, süre, yaş).

In [46]:
temiz = pd.DataFrame(index=pd.Index(list(oyunlar), name="app_id"))
idx = temiz.index

temiz["ad"] = [oyunlar[a]["name"] for a in idx]
temiz["steam_cikis_tarihi"] = pd.to_datetime([oyunlar[a]["release_date"] for a in idx], format="%b %d, %Y")

temiz["fiyat"] = [oyunlar[a]["price"] for a in idx]
temiz["indirim_yuzde"] = pd.Series([int(oyunlar[a]["discount"]) for a in idx], index=idx).astype("Int64")
temiz["normal_fiyat"] = (temiz["fiyat"] / (1 - temiz["indirim_yuzde"] / 100)).astype(float).round(2)

temiz["yas_siniri"] = pd.Series([oyunlar[a]["required_age"] for a in idx], index=idx).astype("Int64")
temiz["sahip_alt"] = pd.Series([int(oyunlar[a]["estimated_owners"].split(" - ")[0]) for a in idx], index=idx).astype("Int64")
temiz["sahip_ust"] = pd.Series([int(oyunlar[a]["estimated_owners"].split(" - ")[1]) for a in idx], index=idx).astype("Int64")
temiz["ortalama_sure_saat"] = [oyunlar[a]["average_playtime_forever"] / 60 for a in idx]
temiz["medyan_sure_saat"] = [oyunlar[a]["median_playtime_forever"] / 60 for a in idx]
temiz["peak_ccu"] = pd.Series([oyunlar[a]["peak_ccu"] for a in idx], index=idx).astype("Int64")

print(temiz.shape)
print(temiz.dtypes)
temiz.loc[["2218750", "1237970", "2357570"]]

(65686, 11)
ad                               str
steam_cikis_tarihi    datetime64[us]
fiyat                        float64
indirim_yuzde                  Int64
normal_fiyat                 float64
yas_siniri                     Int64
sahip_alt                      Int64
sahip_ust                      Int64
ortalama_sure_saat           float64
medyan_sure_saat             float64
peak_ccu                       Int64
dtype: object


,ad,steam_cikis_tarihi,fiyat,indirim_yuzde,normal_fiyat,yas_siniri,sahip_alt,sahip_ust,ortalama_sure_saat,medyan_sure_saat,peak_ccu
app_id,,,,,,,,,,,
2218750,Halls of Torment,2024-09-24,6.66,0,6.66,0,500000,1000000,0.0,0.0,1492
1237970,Titanfall® 2,2020-06-18,4.79,84,29.94,0,5000000,10000000,0.0,0.0,3098
2357570,Overwatch® 2,2023-08-10,0.00,0,0.00,0,0,0,0.0,0.0,0


**Kontrol:** 65.686 × 11. Tarih gerçek tarih tipinde, boş olabilecek tam sayılar Int64.
- normal_fiyat yaklaşık bir değer: games.json indirimi yuvarlıyor (Titanfall 2: 29,94, gerçeği 29,99).
- Overwatch 2'deki 0'lar ve Halls of Torment'in 0 saatlik süresi bir sonraki hücrede NULL olacak.

In [47]:
temiz["istatistik_eksik"] = sahip_00.reindex(idx).astype(bool)
eksik = temiz["istatistik_eksik"]

for sutun in ["indirim_yuzde", "sahip_alt", "sahip_ust", "peak_ccu"]:
    temiz.loc[eksik, sutun] = pd.NA

print("eksik grup:", eksik.sum())
temiz.loc[["2357570", "2218750"]]

eksik grup: 13521


,ad,steam_cikis_tarihi,fiyat,indirim_yuzde,normal_fiyat,yas_siniri,sahip_alt,sahip_ust,ortalama_sure_saat,medyan_sure_saat,peak_ccu,istatistik_eksik
app_id,,,,,,,,,,,,
2357570,Overwatch® 2,2023-08-10,0.00,<NA>,0.00,0,<NA>,<NA>,0.0,0.0,<NA>,True
2218750,Halls of Torment,2024-09-24,6.66,0,6.66,0,500000,1000000,0.0,0.0,1492,False


In [48]:
temiz["yas_siniri"] = temiz["yas_siniri"].mask(temiz["yas_siniri"] == 0)

for sutun in ["ortalama_sure_saat", "medyan_sure_saat"]:
    temiz[sutun] = temiz[sutun].mask(temiz[sutun] == 0)

temiz.isna().sum()

ad                        0
steam_cikis_tarihi        0
fiyat                     0
indirim_yuzde         13521
normal_fiyat              0
yas_siniri            65264
sahip_alt             13521
sahip_ust             13521
ortalama_sure_saat    62231
medyan_sure_saat      62231
peak_ccu              13521
istatistik_eksik          0
dtype: int64

### Uygulama 2: Bilinmeyen değerler NULL
- `istatistik_eksik` sütunu eklendi (owners "0 - 0" olan 13.521 oyun). Bu oyunlarda indirim, sahip sayısı ve peak_ccu NULL.
- Fiyat gerçek, korundu. İndirim bilinmediği için normal_fiyat = fiyat (varsayım).
- yas_siniri 0 → NULL (girilmemiş demek, "sınır yok" değil).
- Oynama süresi 0 → NULL (bilinmiyor).

**Kontrol:** eksik grup 13.521 (birebir). Yaş sınırı sadece 422 oyunda var.
**Dikkat:** oynama süresi sadece 3.455 oyunda (%5) var, 62.231 boş.
Sütun tutuluyor ama agent kuralı: süre boşsa "veri yok" de; süre sıralamalarında kapsamın dar olduğunu söyle.
Test sorusu adayı: "En uzun oynanan RPG'ler hangileri?"

In [49]:
temiz.loc[temiz["ortalama_sure_saat"].notna(), "sahip_alt"].describe()

count            3455.0
mean      337386.396527
std      1837602.767583
min                 0.0
25%             20000.0
50%             50000.0
75%            200000.0
max          50000000.0
Name: sahip_alt, dtype: Float64

**Süre verisi kimde var?** Süresi olan 3.455 oyunun medyan sahip_alt'ı 50.000.
Bütün games.json'da %63'ü "0 - 20.000" iken bu grupta en fazla %25.
Yani süre (SteamSpy kaynaklı) sadece popüler oyunlarda ölçülmüş.
**Fikir:** CSV'deki yorumcu sürelerinden "yorumcu_medyan_sure" hesaplanabilir (23.107 oyun). Yorum temizliğinde ölçülecek.

In [52]:
ok = oy_karsilastirma.reindex(idx)

temiz["olumlu_json"] = ok["json_olumlu"].astype("Int64")
temiz["olumsuz_json"] = ok["json_olumsuz"].astype("Int64")
temiz.loc[eksik, ["olumlu_json", "olumsuz_json"]] = pd.NA

temiz["olumlu_csv"] = ok["csv_olumlu"].fillna(0).astype("Int64")
temiz["olumsuz_csv"] = ok["csv_olumsuz"].fillna(0).astype("Int64")

In [53]:
json_oy = temiz["olumlu_json"].fillna(0) + temiz["olumsuz_json"].fillna(0)
csv_oy = temiz["olumlu_csv"] + temiz["olumsuz_csv"]
json_secilir = json_oy >= csv_oy

temiz["oran_kaynagi"] = "csv"
temiz.loc[json_secilir, "oran_kaynagi"] = "json"
temiz.loc[(json_oy == 0) & (csv_oy == 0), "oran_kaynagi"] = pd.NA

temiz["oy_sayisi"] = json_oy.where(json_secilir, csv_oy)
olumlu = temiz["olumlu_json"].fillna(0).where(json_secilir, temiz["olumlu_csv"])
temiz["olumlu_oran"] = (olumlu / temiz["oy_sayisi"]).mask(temiz["oy_sayisi"] == 0).round(3)

print(temiz["oran_kaynagi"].value_counts(dropna=False))
temiz.loc[["2357570", "1237970", "1222670", "1172470"],
          ["ad", "oy_sayisi", "oran_kaynagi", "olumlu_oran", "olumlu_json", "olumlu_csv"]]

oran_kaynagi
json    35423
csv     15279
NaN     14984
Name: count, dtype: int64


,ad,oy_sayisi,oran_kaynagi,olumlu_oran,olumlu_json,olumlu_csv
app_id,,,,,,
2357570,Overwatch® 2,313125,csv,0.212,<NA>,66448
1237970,Titanfall® 2,237538,json,0.953,226411,12787
1222670,The Sims™ 4,153114,json,0.874,133802,5122
1172470,Apex Legends™,961805,json,0.677,651312,21370


**Kontrol:** kaynak json 35.423, csv 15.279, oy yok 14.984.
Overwatch 2 %21 (CSV'den, json'da istatistik yok), Titanfall 2 %95, Sims 4 %87, Apex %68 (hepsi json'dan).
Sims 4'te CSV'deki düşük oran (%56) seçilmedi; kural doğru çalışıyor.

In [54]:
temiz["yorum_sayisi"] = dosya_ozeti["satir"].reindex(idx).fillna(0).astype("Int64")
temiz["yorum_var"] = temiz["yorum_sayisi"] > 0
temiz["yorum_kapsami"] = (temiz["yorum_sayisi"] / temiz["oy_sayisi"]).mask(temiz["oy_sayisi"] == 0).round(3)

print("yorumu olan:", temiz["yorum_var"].sum())
print("kapsamı %50'den az:", (temiz["yorum_kapsami"] < 0.5).sum())
temiz.loc[["2357570", "1237970", "1222670", "1172470"],
          ["ad", "oy_sayisi", "yorum_sayisi", "yorum_var", "yorum_kapsami"]]

yorumu olan: 23107
kapsamı %50'den az: 27768


,ad,oy_sayisi,yorum_sayisi,yorum_var,yorum_kapsami
app_id,,,,,
2357570,Overwatch® 2,313125,313125,True,1.0
1237970,Titanfall® 2,237538,14029,True,0.059
1222670,The Sims™ 4,153114,9140,True,0.06
1172470,Apex Legends™,961805,50120,True,0.052


### Uygulama 3c: Yorum sütunları
- yorum_sayisi: CSV'deki yorum sayısı (CSV yoksa 0).
- yorum_var: Chroma'da yorumu var mı → agent "yorum yok" diyebilsin.
- yorum_kapsami: yorum_sayisi / oy_sayisi. Düşükse yorumlar az bir kesit (Sims 4 ≈ 0,06); agent oran–yorum çelişkisine karşı uyarır.

In [55]:
((temiz["yorum_kapsami"] < 0.5) & temiz["yorum_var"]).sum()

np.int64(173)

**Kontrol:** yorumu olan 23.107 (= CSV sayısı).
Kapsamı <0,5 olan 27.768 oyunun 27.595'i hiç CSV'si olmayan oyunlar (kapsam 0); CSV'si olup az kapsayan 173 oyun.
Titanfall 2, Sims 4 ve Apex'te yorumlar oyların sadece %5–6'sı → agent bu oyunlarda yorumların sınırlı bir kesit olduğunu söyler.

In [60]:
import html

SPAM_ISARETLERI = ["其他游戏推荐", "友情链接"]

def temiz_metin(metin):
    metin = html.unescape(metin)
    for isaret in SPAM_ISARETLERI:
        if isaret in metin:
            metin = metin.split(isaret)[0]
    return metin.strip()

temiz["kisa_aciklama"] = [temiz_metin(oyunlar[a]["short_description"]) for a in idx]
temiz["oyun_hakkinda"] = [temiz_metin(oyunlar[a]["about_the_game"]) for a in idx]
temiz[["kisa_aciklama", "oyun_hakkinda"]] = temiz[["kisa_aciklama", "oyun_hakkinda"]].replace("", pd.NA)

print(temiz[["kisa_aciklama", "oyun_hakkinda"]].isna().sum())
print(temiz["oyun_hakkinda"].str.len().describe())

kisa_aciklama    5052
oyun_hakkinda    5119
dtype: int64
count    60567.000000
mean      1267.731917
std       1005.204706
min          3.000000
25%        652.000000
50%       1033.000000
75%       1608.000000
max      67520.000000
Name: oyun_hakkinda, dtype: float64


**Düzeltme:** "友情链接" (dost bağlantıları) başlıklı ikinci bir spam türü bulundu (Crazy Zombies, 95.281 karakter).
temiz_metin artık bir başlık listesine bakıyor: SPAM_ISARETLERI. Chroma'da 2.000 karakterden kesilecek oyun: 9.160.

In [57]:
temiz["iskelet"] = [
    not oyunlar[a]["about_the_game"] and not oyunlar[a]["genres"]
    and not oyunlar[a]["supported_languages"] and not oyunlar[a]["categories"]
    for a in idx
]
temiz["yetiskin"] = ok["yetiskin"]
temiz["yetiskin_etiket_sayisi"] = ok["yetiskin_sayisi"].astype("Int64")
temiz["dil_spam"] = [len(oyunlar[a]["supported_languages"]) >= 100 for a in idx]
temiz["ses_spam"] = [len(oyunlar[a]["full_audio_languages"]) >= 50 for a in idx]

temiz[["iskelet", "yetiskin", "dil_spam", "ses_spam"]].sum()

iskelet     4817
yetiskin     536
dil_spam    1160
ses_spam     916
dtype: int64

In [59]:
(temiz["oyun_hakkinda"].str.len() > 2000).sum()

np.int64(9160)

**Kontrol:** bayraklar birebir: iskelet 4.817, yetişkin 536, dil_spam 1.160, ses_spam 916.
Boş açıklama ~5.100 (4.817 iskelet + ~300 açıklamasız oyun).
Açıklama medyanı 1.033 karakter, %75'i 1.608'in altında → Chroma'daki 2.000 kesmesi çoğu oyuna dokunmaz.
En uzun metin 95.281 karakter → incelendi (aşağıda).

In [61]:
uzunluk = temiz["oyun_hakkinda"].str.len().sort_values(ascending=False)
pd.DataFrame({"ad": temiz.loc[uzunluk.index[:10], "ad"], "uzunluk": uzunluk[:10]})

,ad,uzunluk
app_id,,
1493800,Aircraft Carrier Survival: Tutorial,67520.0
2329760,Vampire Memories - 吸血鬼的回忆,46716.0
2459790,逆乱水浒之山贼王,20970.0
1185680,White Wings ホワイトウィングス,16986.0
1405960,VR Travelling in 18th-19th Century Europe,16626.0
2140680,Code R.U.B.I.K.,16362.0
2081600,THE STARFIGHTER,16231.0
2348820,Living in a Brothel,15952.0
2398720,Rune in the Three Kingdoms,15626.0


In [62]:
for a in ["1493800", "2329760"]:
    metin = temiz.loc[a, "oyun_hakkinda"]
    print(a, temiz.loc[a, "ad"])
    print(metin[:1000])
    print("...")
    print(metin[-1000:])
    print("=" * 80)

1493800 Aircraft Carrier Survival: Tutorial
[, , , ,Take control of the aircraft carrier, its crew, officers and planes during WW2. Give an example of military prowess by choosing the right equipment, crew, and escort for the mission. Remember that your success will depend greatly on your flexibility in managing the available fleet resources. , , ,Assign a crew and choose what your officers should focus on. Your ship has many stations to man, and proper management will be the key to executing your plan, whether by taking on an enemy directly, sneaking past them, or taking an alternative route to completely avoid contact. , , ,Your damage control teams can handle fire outbreaks, torpedo strikes, hull flooding, enemy bombardment, airstrikes, and suicide attacks from enemy planes, but not all of them at once. Assess which threats need to be dealt with first in order to survive. , , , ,Use maneuvers, aircraft, anti-aircraft turrets, and whatever else you can to survive the enemy attack. Ma

**Kontrol:** Crazy Zombies temizlendi, max 95.281 → 67.520. En uzun 10 metin görsel roman/eğitim türü uzun açıklamalar.
Spam'ler metnin sonuna ekleniyor; Chroma'ya ilk 2.000 karakter girdiği için kalan spam riski düşük.

In [63]:
virgul = temiz["oyun_hakkinda"].str.contains(r",\s*,\s*,", na=False)
url = temiz["oyun_hakkinda"].str.contains(r"\[/?url", na=False)
print("virgül artığı:", virgul.sum())
print("url kodu:", url.sum())

virgül artığı: 8
url kodu: 34


**En uzun 2 metin:** Aircraft Carrier Survival = bozuk kopyalanmış metin (tekrarlanan paragraflar, ", , ," artıkları, [/url]); Vampire Memories = açıklamaya eklenmiş geliştirme günlüğü (gerçek metin).
Artıkların kaç oyunda olduğu ölçülüyor.

**Artık ölçümü:** ", , ," 8 oyunda, [url] kodu 34 oyunda (açıklamalı oyunların < %0,1). Temizlenmedi; etkisi ihmal edilebilir.

In [64]:
oyun_tur = pd.DataFrame(
    [(a, tur) for a in idx for tur in oyunlar[a]["genres"]],
    columns=["app_id", "tur"])

etiket_satirlari = []
for a in idx:
    etiketler = oyunlar[a]["tags"]
    if not isinstance(etiketler, dict):
        continue
    for sira, (etiket, oy) in enumerate(etiketler.items(), start=1):
        etiket_satirlari.append((a, etiket, oy, sira))
oyun_etiket = pd.DataFrame(etiket_satirlari, columns=["app_id", "etiket", "oy", "sira"])

print("tür:", oyun_tur.shape, "oyun:", oyun_tur["app_id"].nunique())
print("etiket:", oyun_etiket.shape, "oyun:", oyun_etiket["app_id"].nunique())
print("sıra bozuk:", (oyun_etiket.groupby("app_id")["oy"].diff() > 0).sum())
oyun_etiket[oyun_etiket["app_id"] == "1237970"]

tür: (176611, 2) oyun: 60586
etiket: (734422, 4) oyun: 44786
sıra bozuk: 4


,app_id,etiket,oy,sira
651755,1237970,FPS,1206,1
651756,1237970,Mechs,987,2
651757,1237970,Multiplayer,980,3
651758,1237970,Shooter,945,4
651759,1237970,Singleplayer,895,5
651760,1237970,Action,890,6
651761,1237970,Parkour,872,7
651762,1237970,Sci-fi,831,8
651763,1237970,First-Person,726,9
651764,1237970,Futuristic,701,10


### Uygulama 5: Yardımcı tablolar
- oyun_tur(app_id, tur): her satır bir oyun + bir tür.
- oyun_etiket(app_id, etiket, oy, sira): sira = etiketin oyundaki sırası (1 = en çok oy alan). Etiketsiz oyunlar bu tabloda yok.
- Neden ayrı tablo: "FPS oyunlarını getir" gibi SQL sorguları basit bir filtre olsun diye.

In [65]:
bozuk = oyun_etiket.groupby("app_id")["oy"].diff() > 0
bozuk_oyunlar = oyun_etiket.loc[bozuk, "app_id"]
oyun_etiket[oyun_etiket["app_id"].isin(bozuk_oyunlar)]

,app_id,etiket,oy,sira
2913,1144250,Early Access,23,1
2914,1144250,Nudity,41,2
2915,1144250,Sexual Content,40,3
2916,1144250,Gore,35,4
2917,1144250,Violent,34,5
...,...,...,...,...
613864,1168170,Atmospheric,11,15
613865,1168170,Singleplayer,11,16
613866,1168170,Short,11,17
613867,1168170,Dark,11,18


**Kontrol:** tür 176.611 satır / 60.586 oyun; etiket 734.422 satır / 44.786 oyun (= etiketli oyun sayısı, ort. ~16 etiket).
Etiket sırası games.json'da oy sayısına göre; sadece 4 satırda bozuk (inceleniyor).
Titanfall 2: FPS, Mechs, Multiplayer... → etiketler türlerden daha ayrıntılı ("Mechs", "Parkour" türlerde yok).

In [66]:
oyun_etiket = oyun_etiket.sort_values(["app_id", "oy"], ascending=[True, False], kind="stable")
oyun_etiket["sira"] = oyun_etiket.groupby("app_id").cumcount() + 1

print("sıra bozuk:", (oyun_etiket.groupby("app_id")["oy"].diff() > 0).sum())
oyun_etiket[oyun_etiket["app_id"] == "1144250"].head(6)

sıra bozuk: 0


,app_id,etiket,oy,sira
2914,1144250,Nudity,41,1
2915,1144250,Sexual Content,40,2
2916,1144250,Gore,35,3
2917,1144250,Violent,34,4
2918,1144250,Action,33,5
2913,1144250,Early Access,23,6


In [67]:
ilk5 = oyun_etiket[oyun_etiket["sira"] <= 5]
yetiskin_sayim = ilk5[ilk5["etiket"].isin(YETISKIN_ETIKETLERI)].groupby("app_id").size()
print("yetişkin (yeni sıra):", (yetiskin_sayim >= 2).sum())

yetişkin (yeni sıra): 536


**Etiket sırası:** 4 oyunda games.json sırası oya göre değil (Steam "Early Access" gibi etiketleri başa koyuyor, örn. 1144250).
Karar: sira oy sayısına göre yeniden hesaplandı (eşitlikte games.json sırası). Yetişkin bayrağı yeni sırayla tekrar kontrol edildi.

**Kontrol:** sira oya göre yeniden hesaplandı → sıra bozuk 0. 1144250'de Early Access 1. sıradan 6. sıraya indi.
Yetişkin bayrağı yeni sırayla da 536 → değişiklik yok.

In [68]:
oyun_etiket.groupby("app_id").size().describe()

count    44786.000000
mean        16.398473
std          4.740898
min          1.000000
25%         13.000000
50%         19.000000
75%         20.000000
max         20.000000
dtype: float64

**Kavramlar:** etiket = oyuncuların eklediği kelime; etiketin oyu = o etiketi ekleyen oyuncu sayısı (beğeni değil); sira = oyunda en çok eklenen etiket 1.
Tür = geliştiricinin seçtiği, daha kaba sınıf.
**Etiket sayısı:** medyan 19, çoğu oyun 20 (Steam sınırı); en az 1. Az etiketli/az oylu oyunlarda etiketler daha az güvenilir.

In [69]:
kategori_satirlari = []
for a in idx:
    kategoriler = {"VR Supported" if k == "VR Support" else k for k in oyunlar[a]["categories"]}
    kategori_satirlari += [(a, k) for k in sorted(kategoriler)]
oyun_kategori = pd.DataFrame(kategori_satirlari, columns=["app_id", "kategori"])

dil_satirlari = []
for a in idx:
    sesli = set(oyunlar[a]["full_audio_languages"])
    diller = set(oyunlar[a]["supported_languages"]) | sesli
    dil_satirlari += [(a, d, d in sesli) for d in sorted(diller)]
oyun_dil = pd.DataFrame(dil_satirlari, columns=["app_id", "dil", "ses_var"])

print("kategori:", oyun_kategori.shape, "oyun:", oyun_kategori["app_id"].nunique(),
      "farklı kategori:", oyun_kategori["kategori"].nunique())
print("dil:", oyun_dil.shape, "oyun:", oyun_dil["app_id"].nunique())

spamsiz = oyun_dil[~oyun_dil["app_id"].map(temiz["dil_spam"])]
tr = spamsiz[spamsiz["dil"] == "Turkish"]
print("Türkçe metin (spam hariç):", len(tr), "| Türkçe ses:", tr["ses_var"].sum())

kategori: (235694, 2) oyun: 60374 farklı kategori: 39
dil: (345070, 3) oyun: 60562
Türkçe metin (spam hariç): 4779 | Türkçe ses: 1133


In [70]:
yazilim_idleri = oy_karsilastirma.index[oy_karsilastirma["yazilim"]]
oyunlar_son = temiz.drop(yazilim_idleri)
kalan = set(oyunlar_son.index)

tablolar = {
    "oyunlar": oyunlar_son,
    "oyun_tur": oyun_tur[oyun_tur["app_id"].isin(kalan)],
    "oyun_etiket": oyun_etiket[oyun_etiket["app_id"].isin(kalan)],
    "oyun_kategori": oyun_kategori[oyun_kategori["app_id"].isin(kalan)],
    "oyun_dil": oyun_dil[oyun_dil["app_id"].isin(kalan)],
}

for ad, tablo in tablolar.items():
    tablo.to_parquet(ISLENMIS / f"{ad}.parquet", index=(ad == "oyunlar"))
    print(f"{ad}: {tablo.shape}")

kontrol = pd.read_parquet(ISLENMIS / "oyunlar.parquet")
print("geri okunan:", kontrol.shape, "| tipler aynı mı:", (kontrol.dtypes == oyunlar_son.dtypes).all())

oyunlar: (64948, 29)
oyun_tur: (174563, 2)
oyun_etiket: (730709, 4)
oyun_kategori: (235168, 2)
oyun_dil: (341659, 3)
geri okunan: (64948, 29) | tipler aynı mı: True


### Uygulama 6: Kayıt
738 yazılım atıldı → 64.948 oyun. Yardımcı tablolar da aynı oyunlara süzüldü.
data/processed/ altına kaydedildi: oyunlar, oyun_tur, oyun_etiket, oyun_kategori, oyun_dil (.parquet).
Geri okuma kontrolü: boyut ve tipler korunuyor. games.json temizliği tamam; 04'te SQLite bu dosyalardan kurulacak.

**Sonuç:** oyunlar 64.948 × 29; oyun_tur 174.563, oyun_etiket 730.709, oyun_kategori 235.168, oyun_dil 341.659 satır.
Parquet'ten geri okuma: boyut ve tipler birebir. games.json temizliği tamamlandı.
Eksik: ilk/son_yorum_tarihi → yorum temizliğinde eklenecek.